# 03 Data Integration and Final Dataset Preparation

This notebook builds the cleaned dataset for the **30-minute-ahead LOS prediction task**.

## Scope

This notebook is limited to data preparation and quality assurance. It does **not** train or tune machine-learning models.

## Inputs

The notebook reads the preprocessed Parquet files from `data/processed/`:

- `train_preprocessed.parquet`
- `segments_preprocessed.parquet`
- `streets_preprocessed.parquet`
- `nodes_preprocessed.parquet`
- `weather_preprocessed.parquet`
- `segment_status_preprocessed.parquet`

## Outputs

The final release is written to `data/processed/model_v2/`:

- `train_dataset_v2.parquet`
- `holdout_dataset_v2.parquet`
- `train_metadata_v2.parquet`
- `holdout_metadata_v2.parquet`
- `feature_schema_v2.json`
- `data_quality_report_v2.md`

Run the notebook from top to bottom after restarting the kernel. The notebook locates the project root from the current working directory.


## 1. Load preprocessed source tables

Load the six preprocessed tables. The project root is detected by locating `data/processed/`.


In [1]:

from pathlib import Path
import pandas as pd
import numpy as np

# Find project root
PROJECT_ROOT = Path.cwd()

while not (PROJECT_ROOT / "data" / "processed").exists():
    if PROJECT_ROOT == PROJECT_ROOT.parent:
        raise FileNotFoundError("Project root not found")
    PROJECT_ROOT = PROJECT_ROOT.parent

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

# Load preprocessed datasets
train = pd.read_parquet(
    PROCESSED_DIR / "train_preprocessed.parquet"
)

segments = pd.read_parquet(
    PROCESSED_DIR / "segments_preprocessed.parquet"
)

streets = pd.read_parquet(
    PROCESSED_DIR / "streets_preprocessed.parquet"
)

nodes = pd.read_parquet(
    PROCESSED_DIR / "nodes_preprocessed.parquet"
)

weather = pd.read_parquet(
    PROCESSED_DIR / "weather_preprocessed.parquet"
)

status = pd.read_parquet(
    PROCESSED_DIR / "segment_status_preprocessed.parquet"
)

# Display dataset information
datasets = {
    "train": train,
    "segments": segments,
    "streets": streets,
    "nodes": nodes,
    "weather": weather,
    "status": status
}

for name, df in datasets.items():
    print(f"{name}: {df.shape}")

print("\nALL DATASETS LOADED SUCCESSFULLY!")


train: (33441, 30)
segments: (84633, 11)
streets: (5553, 5)
nodes: (577967, 3)
weather: (7056, 11)
status: (90938, 5)

ALL DATASETS LOADED SUCCESSFULLY!


## 2. Validate source keys and target timestamps

Check primary keys, duplicate segment-time observations, LOS values, and the 30-minute interval already represented in the preprocessed training table.


In [2]:

# 1. Check primary keys
print("PRIMARY KEY VALIDATION")

print(
    "Duplicate segment IDs:",
    segments["_id"].duplicated().sum()
)

print(
    "Duplicate node IDs:",
    nodes["_id"].duplicated().sum()
)

print(
    "Duplicate street IDs:",
    streets["_id"].duplicated().sum()
)

assert segments["_id"].is_unique
assert nodes["_id"].is_unique
assert streets["_id"].is_unique

# 2. Check train observation keys
duplicate_train_keys = train.duplicated(
    ["segment_id", "observation_time_utc"]
).sum()

print("\nTRAIN KEY VALIDATION")
print("Duplicate segment-time keys:", duplicate_train_keys)

assert duplicate_train_keys == 0

# 3. Check timestamps
assert train["observation_time_utc"].notna().all()
assert train["target_time_utc"].notna().all()

# 4. Check target time interval
time_difference = (
    train["target_time_utc"]
    - train["observation_time_utc"]
)

print("\nTARGET INTERVAL VALIDATION")
display(time_difference.value_counts())

assert (
    time_difference == pd.Timedelta(minutes=30)
).all(), "Unexpected target interval!"

# 5. Check LOS classes
print("\nLOS DISTRIBUTION")

display(
    train["LOS"].value_counts(dropna=False)
)

print("\nVALIDATION COMPLETED!")


PRIMARY KEY VALIDATION
Duplicate segment IDs: 0
Duplicate node IDs: 0
Duplicate street IDs: 0

TRAIN KEY VALIDATION
Duplicate segment-time keys: 0

TARGET INTERVAL VALIDATION


0 days 00:30:00    33441
Name: count, dtype: int64


LOS DISTRIBUTION


LOS
A    13278
B     4700
F     4089
C     3839
E     3828
D     3707
Name: count, dtype: int64


VALIDATION COMPLETED!


## 3. Build road features from segments and nodes

Join each road segment to its start and end nodes, then derive midpoint coordinates. The original source `length` is retained for the subsequent precision audit.


In [3]:

# 1. Select required segment attributes
road = segments[
    [
        "_id",
        "s_node_id",
        "e_node_id",
        "length",
        "street_level",
        "street_type"
    ]
].copy()

road = road.rename(
    columns={"_id": "segment_id"}
)

# 2. Prepare start node coordinates
start_nodes = nodes[
    ["_id", "long", "lat"]
].rename(
    columns={
        "_id": "s_node_id",
        "long": "start_longitude",
        "lat": "start_latitude"
    }
)

# 3. Prepare end node coordinates
end_nodes = nodes[
    ["_id", "long", "lat"]
].rename(
    columns={
        "_id": "e_node_id",
        "long": "end_longitude",
        "lat": "end_latitude"
    }
)

# 4. Join start node
road = road.merge(
    start_nodes,
    on="s_node_id",
    how="left",
    validate="many_to_one",
    indicator="_start_match"
)

unmatched_start_nodes = (
    road["_start_match"] == "left_only"
).sum()

print("Unmatched start nodes:", unmatched_start_nodes)
assert unmatched_start_nodes == 0

road = road.drop(columns="_start_match")

# 5. Join end node
road = road.merge(
    end_nodes,
    on="e_node_id",
    how="left",
    validate="many_to_one",
    indicator="_end_match"
)

unmatched_end_nodes = (
    road["_end_match"] == "left_only"
).sum()

print("Unmatched end nodes:", unmatched_end_nodes)
assert unmatched_end_nodes == 0

road = road.drop(columns="_end_match")

# 6. Create geographic features
road["center_longitude"] = (
    road["start_longitude"]
    + road["end_longitude"]
) / 2

road["center_latitude"] = (
    road["start_latitude"]
    + road["end_latitude"]
) / 2

# 7. Identify zero-length roads
road["length_zero_flag"] = (
    road["length"] == 0
).astype("int8")

# 8. Validate road table
assert road["segment_id"].is_unique
assert pd.to_numeric(road["length"], errors="coerce").notna().all()
assert pd.to_numeric(road["length"], errors="coerce").ge(0).all()

print("\nROAD FEATURE TABLE")

print("Shape:", road.shape)

print(
    "Duplicate segment IDs:",
    road["segment_id"].duplicated().sum()
)

print(
    "Missing coordinates:",
    road[
        [
            "start_longitude",
            "start_latitude",
            "end_longitude",
            "end_latitude"
        ]
    ].isna().any(axis=1).sum()
)

display(road.head(10))

print("\nROAD FEATURE INTEGRATION COMPLETED!")


Unmatched start nodes: 0
Unmatched end nodes: 0

ROAD FEATURE TABLE
Shape: (84633, 13)
Duplicate segment IDs: 0
Missing coordinates: 0


,segment_id,s_node_id,e_node_id,length,street_level,street_type,start_longitude,start_latitude,end_longitude,end_latitude,center_longitude,center_latitude,length_zero_flag
0,0,373543511,5468660805,114,1,trunk,106.601780,10.727718,106.601621,10.726701,106.601700,10.727209,0
1,1,5468660805,5738158916,9,1,trunk,106.601621,10.726701,106.601607,10.726613,106.601614,10.726657,0
2,2,5738158916,5738158918,23,1,trunk,106.601607,10.726613,106.601574,10.726401,106.601590,10.726507,0
3,3,5738158918,5738158912,66,1,trunk,106.601574,10.726401,106.601481,10.725809,106.601527,10.726105,0
4,4,5738158912,5758104203,127,1,trunk,106.601481,10.725809,106.601277,10.724676,106.601379,10.725242,0
5,5,5758104203,2242413001,22,1,trunk,106.601277,10.724676,106.601241,10.724474,106.601259,10.724575,0
6,6,2242413001,4066528708,15,1,trunk,106.601241,10.724474,106.601241,10.724338,106.601241,10.724406,0
7,7,4066528708,4066528709,15,1,trunk,106.601241,10.724338,106.601305,10.724210,106.601273,10.724274,0
8,8,4066528709,4066528710,10,1,trunk,106.601305,10.724210,106.601353,10.724132,106.601329,10.724171,0
9,9,4066528710,2242412994,13,1,trunk,106.601353,10.724132,106.601431,10.724037,106.601392,10.724084,0



ROAD FEATURE INTEGRATION COMPLETED!


## 4. Integrate road attributes into traffic observations

The road join must preserve the number of traffic observations and the unique segment-time key.


In [4]:

# Preserve original number of observations
original_train_rows = len(train)

# Join road features
train_road = train.merge(
    road,
    on="segment_id",
    how="left",
    validate="many_to_one",
    indicator=True
)

# Check matching results
print("ROAD INTEGRATION RESULTS")

print("Original train rows:", original_train_rows)
print("Integrated rows:", len(train_road))

print("\nMatch status:")
display(train_road["_merge"].value_counts())

# Calculate unmatched percentage
unmatched = (
    train_road["_merge"] == "left_only"
).sum()

coverage = (
    (len(train_road) - unmatched)
    / len(train_road) * 100
)

print(f"\nRoad coverage: {coverage:.2f}%")
assert unmatched == 0, "Some traffic observations have no road-feature match."

# Remove merge indicator
train_road = train_road.drop(
    columns="_merge"
)

# Validate
assert len(train_road) == original_train_rows

assert not train_road.duplicated(
    ["segment_id", "observation_time_utc"]
).any()

assert train_road["center_longitude"].notna().all()
assert train_road["center_latitude"].notna().all()

print("\nROAD INTEGRATION COMPLETED!")
print("Final shape:", train_road.shape)


ROAD INTEGRATION RESULTS
Original train rows: 33441
Integrated rows: 33441

Match status:


_merge
both          33441
left_only         0
right_only        0
Name: count, dtype: int64


Road coverage: 100.00%

ROAD INTEGRATION COMPLETED!
Final shape: (33441, 42)


## 5. Integrate historical weather

Use a backward as-of join with a maximum weather age of 30 minutes. This ensures that a traffic observation is matched only to weather data at or before the observation time.


In [5]:

# 1. Select independent weather features
weather_features = [
    "temperature_2m",
    "relative_humidity_2m",
    "precipitation",
    "weather_code",
    "wind_speed_10m",
    "cloud_cover"
]

weather_lookup = weather[
    ["weather_time_utc"] + weather_features
].copy()

# 2. Sort both datasets by timestamp
train_road_sorted = train_road.sort_values(
    "observation_time_utc"
).copy()

weather_lookup = weather_lookup.sort_values(
    "weather_time_utc"
).copy()

# 3. Backward temporal join
integrated = pd.merge_asof(
    train_road_sorted,
    weather_lookup,
    left_on="observation_time_utc",
    right_on="weather_time_utc",
    direction="backward",
    tolerance=pd.Timedelta(minutes=30),
    allow_exact_matches=True
)

# 4. Calculate weather age
integrated["weather_age_minutes"] = (
    integrated["observation_time_utc"]
    - integrated["weather_time_utc"]
).dt.total_seconds() / 60

# 5. Evaluate coverage
weather_matched = (
    integrated["weather_time_utc"].notna()
)

matched_count = weather_matched.sum()

weather_coverage = (
    matched_count / len(integrated) * 100
)

print("WEATHER INTEGRATION RESULTS")

print("Total observations:", len(integrated))
print("Matched weather:", matched_count)
print(f"Weather coverage: {weather_coverage:.2f}%")

print("\nWeather age statistics:")
display(
    integrated["weather_age_minutes"].describe()
)

print("\nMissing weather features:")
display(
    integrated[weather_features].isna().sum()
)

# 6. Validate temporal correctness
assert (
    integrated["weather_age_minutes"].dropna() >= 0
).all()

assert (
    integrated["weather_age_minutes"].dropna() <= 30
).all()

assert weather_matched.all(), "Some observations have no weather match."
assert integrated[weather_features].notna().all().all()

assert len(integrated) == len(train_road)

print("\nWEATHER INTEGRATION COMPLETED!")
print("Integrated shape:", integrated.shape)


WEATHER INTEGRATION RESULTS
Total observations: 33441
Matched weather: 33441
Weather coverage: 100.00%

Weather age statistics:


count    33441.000000
mean        15.182560
std         14.999113
min          0.000000
25%          0.000000
50%         30.000000
75%         30.000000
max         30.000000
Name: weather_age_minutes, dtype: float64


Missing weather features:


temperature_2m          0
relative_humidity_2m    0
precipitation           0
weather_code            0
wind_speed_10m          0
cloud_cover             0
dtype: int64


WEATHER INTEGRATION COMPLETED!
Integrated shape: (33441, 50)


## 6. Generate the future LOS target

For each observation, match the same road segment at exactly `observation_time_utc + 30 minutes`. Rows without an exact future LOS match remain outside the supervised target dataset.


In [6]:

# 1. Create future LOS lookup
target_lookup = train[
    [
        "segment_id",
        "observation_time_utc",
        "LOS"
    ]
].copy()

target_lookup = target_lookup.rename(
    columns={
        "observation_time_utc": "target_time_utc",
        "LOS": "target_los_30m"
    }
)

# 2. Verify lookup key uniqueness
assert not target_lookup.duplicated(
    ["segment_id", "target_time_utc"]
).any()

# 3. Exact join using segment and target timestamp
integrated = integrated.merge(
    target_lookup,
    on=["segment_id", "target_time_utc"],
    how="left",
    validate="many_to_one",
    indicator="_target_match"
)

# 4. Evaluate target coverage
target_matched = (
    integrated["_target_match"] == "both"
)

valid_target_count = target_matched.sum()

target_coverage = (
    valid_target_count / len(integrated) * 100
)

print("TARGET GENERATION RESULTS")

print("Original observations:", len(integrated))
print("Valid future LOS:", valid_target_count)

print(f"Target coverage: {target_coverage:.2f}%")

print("\nTarget matching:")
display(
    integrated["_target_match"].value_counts()
)

# 5. Target class distribution
print("\nFuture LOS distribution:")

display(
    integrated.loc[
        target_matched,
        "target_los_30m"
    ].value_counts().sort_index().to_frame("Count")
)

# 6. Remove temporary matching indicator
integrated = integrated.drop(
    columns="_target_match"
)

# 7. Create modeling dataset
model_dataset = integrated.loc[
    integrated["target_los_30m"].notna()
].copy()

# 8. Validate target dataset
assert model_dataset["target_los_30m"].notna().all()

assert not model_dataset.duplicated(
    ["segment_id", "observation_time_utc"]
).any()

print("\nMODEL DATASET CREATED")

print("Shape:", model_dataset.shape)

print("\nTarget labels:")
display(
    model_dataset["target_los_30m"].value_counts().sort_index()
)


TARGET GENERATION RESULTS
Original observations: 33441
Valid future LOS: 7898
Target coverage: 23.62%

Target matching:


_target_match
left_only     25543
both           7898
right_only        0
Name: count, dtype: int64


Future LOS distribution:


,Count
target_los_30m,
A,2591
B,1167
C,837
D,1740
E,1100
F,463



MODEL DATASET CREATED
Shape: (7898, 51)

Target labels:


target_los_30m
A    2591
B    1167
C     837
D    1740
E    1100
F     463
Name: count, dtype: int64

## 7. Audit target coverage and selection patterns

The following audits describe where valid future labels are available. They are diagnostic only; they do not change the target labels or rebalance the data.


In [7]:

# 1. Identify observations with valid future LOS
audit = integrated.copy()

audit["has_target"] = (
    audit["target_los_30m"].notna()
)

print("[OVERALL TARGET COVERAGE]")

print("Total observations:", len(audit))
print("Valid targets:", audit["has_target"].sum())

print(
    "Coverage (%):",
    round(audit["has_target"].mean() * 100, 2)
)

# 2. Coverage by segment

segment_coverage = (
    audit.groupby("segment_id")
    .agg(
        total_observations=("has_target", "size"),
        valid_targets=("has_target", "sum")
    )
)

segment_coverage["coverage_percent"] = (
    segment_coverage["valid_targets"]
    / segment_coverage["total_observations"]
    * 100
)

print("\n[COVERAGE BY SEGMENT]")

display(
    segment_coverage["coverage_percent"].describe()
)

print(
    "Total segments:",
    len(segment_coverage)
)

print(
    "Segments without target:",
    (
        segment_coverage["valid_targets"] == 0
    ).sum()
)

print(
    "Segments with at least one target:",
    (
        segment_coverage["valid_targets"] > 0
    ).sum()
)

# 3. Coverage by current LOS

print("\n[COVERAGE BY CURRENT LOS]")

los_coverage = (
    audit.groupby("LOS", observed=True)
    .agg(
        total=("has_target", "size"),
        valid_targets=("has_target", "sum")
    )
)

los_coverage["coverage_percent"] = (
    los_coverage["valid_targets"]
    / los_coverage["total"]
    * 100
)

display(los_coverage)


# 4. Coverage by local hour

print("\n[COVERAGE BY LOCAL HOUR]")

hour_coverage = (
    audit.groupby("hour_local", observed=True)
    .agg(
        total=("has_target", "size"),
        valid_targets=("has_target", "sum")
    )
)

hour_coverage["coverage_percent"] = (
    hour_coverage["valid_targets"]
    / hour_coverage["total"]
    * 100
)

display(hour_coverage)


# 5. Coverage by month

print("\n[COVERAGE BY LOCAL MONTH]")

month_coverage = (
    audit.groupby("month_local", observed=True)
    .agg(
        total=("has_target", "size"),
        valid_targets=("has_target", "sum")
    )
)

month_coverage["coverage_percent"] = (
    month_coverage["valid_targets"]
    / month_coverage["total"]
    * 100
)

display(month_coverage)


# 6. Current LOS vs Future LOS

print("\n[CURRENT LOS VS FUTURE LOS]")

transition_matrix = pd.crosstab(
    model_dataset["LOS"],
    model_dataset["target_los_30m"],
    normalize="index"
).round(3)

display(transition_matrix)

print("\nSELECTION BIAS AUDIT COMPLETED!")


[OVERALL TARGET COVERAGE]
Total observations: 33441
Valid targets: 7898
Coverage (%): 23.62

[COVERAGE BY SEGMENT]


count    10027.000000
mean        19.688884
std         23.058319
min          0.000000
25%          0.000000
50%          0.000000
75%         50.000000
max         83.333333
Name: coverage_percent, dtype: float64

Total segments: 10027
Segments without target: 5420
Segments with at least one target: 4607

[COVERAGE BY CURRENT LOS]


,total,valid_targets,coverage_percent
LOS,,,
A,13278,2762,20.801326
B,4700,1330,28.297872
C,3839,1141,29.721282
D,3707,502,13.541948
E,3828,1218,31.818182
F,4089,945,23.110785



[COVERAGE BY LOCAL HOUR]


,total,valid_targets,coverage_percent
hour_local,,,
0,20,3,15.000000
1,10,0,0.000000
2,5,0,0.000000
3,2,0,0.000000
4,1,1,100.000000
5,108,8,7.407407
6,4174,3645,87.326306
7,5390,40,0.742115
8,1582,47,2.970923



[COVERAGE BY LOCAL MONTH]


,total,valid_targets,coverage_percent
month_local,,,
1,2802,118,4.211278
2,1,0,0.000000
3,1743,30,1.721170
4,1229,34,2.766477
7,44,12,27.272727
8,7298,3624,49.657440
9,204,1,0.490196
10,13,0,0.000000
11,4762,398,8.357833



[CURRENT LOS VS FUTURE LOS]


target_los_30m,A,B,C,D,E,F
LOS,,,,,,
A,0.927,0.006,0.014,0.030,0.019,0.004
B,0.013,0.819,0.048,0.071,0.047,0.002
C,0.004,0.032,0.223,0.465,0.266,0.009
D,0.000,0.014,0.223,0.478,0.269,0.016
E,0.001,0.010,0.202,0.461,0.309,0.018
F,0.011,0.005,0.127,0.243,0.179,0.435



SELECTION BIAS AUDIT COMPLETED!


In [8]:

# 1. Create audit dataset
audit = integrated.copy()

# IMPORTANT: Recreate target indicator
audit["has_target"] = (
    audit["target_los_30m"].notna()
)

# 2. Ensure datetime format
audit["observation_time_local"] = pd.to_datetime(
    audit["observation_time_local"]
)

# 3. Create date and month identifiers
audit["local_date"] = (
    audit["observation_time_local"]
    .dt.strftime("%Y-%m-%d")
)

audit["year_month"] = (
    audit["observation_time_local"]
    .dt.strftime("%Y-%m")
)


# 4. Coverage by date

daily_coverage = (
    audit.groupby("local_date")
    .agg(
        total_observations=("has_target", "size"),
        valid_targets=("has_target", "sum")
    )
)

daily_coverage["coverage_percent"] = (
    daily_coverage["valid_targets"]
    / daily_coverage["total_observations"]
    * 100
)

print("DAILY COVERAGE SUMMARY")

print("Number of unique dates:", len(daily_coverage))

display(
    daily_coverage["coverage_percent"].describe()
)


# 5. Dates with most observations

print("\nTOP 20 DATES BY OBSERVATIONS")

display(
    daily_coverage.sort_values(
        "total_observations",
        ascending=False
    ).head(20)
)


# 6. Dates with most valid targets

print("\nTOP 20 DATES BY VALID TARGETS")

display(
    daily_coverage.sort_values(
        "valid_targets",
        ascending=False
    ).head(20)
)


# 7. Monthly coverage

monthly_coverage = (
    audit.groupby("year_month")
    .agg(
        total_observations=("has_target", "size"),
        valid_targets=("has_target", "sum")
    )
)

monthly_coverage["coverage_percent"] = (
    monthly_coverage["valid_targets"]
    / monthly_coverage["total_observations"]
    * 100
)

print("\nMONTHLY COVERAGE")

display(monthly_coverage)


# 8. Target count by month and hour

valid_audit = audit.loc[
    audit["has_target"]
].copy()

target_hour_month = pd.crosstab(
    valid_audit["year_month"],
    valid_audit["hour_local"]
)

print("\nTARGET COUNT BY MONTH AND HOUR")

display(target_hour_month)


# 9. Temporal range

print("\nTEMPORAL RANGE")

print(
    "First observation:",
    audit["observation_time_local"].min()
)

print(
    "Last observation:",
    audit["observation_time_local"].max()
)

print(
    "Unique target dates:",
    valid_audit["local_date"].nunique()
)

print(
    "Unique target timestamps:",
    valid_audit["observation_time_utc"].nunique()
)

print("\nDAILY TEMPORAL AUDIT COMPLETED!")


DAILY COVERAGE SUMMARY
Number of unique dates: 120


count    120.000000
mean      12.591913
std       19.469430
min        0.000000
25%        0.000000
50%        1.904817
75%       16.597655
max       83.333333
Name: coverage_percent, dtype: float64


TOP 20 DATES BY OBSERVATIONS


,total_observations,valid_targets,coverage_percent
local_date,,,
2020-08-03,7223,3611,49.993078
2020-12-12,1601,448,27.982511
2020-12-06,1515,457,30.165017
2020-11-29,1283,116,9.041309
2020-12-01,1246,173,13.884430
2020-12-24,1195,64,5.355649
2020-12-28,1114,359,32.226212
2020-12-26,978,461,47.137014
2020-11-30,977,157,16.069601



TOP 20 DATES BY VALID TARGETS


,total_observations,valid_targets,coverage_percent
local_date,,,
2020-08-03,7223,3611,49.993078
2020-12-26,978,461,47.137014
2020-12-06,1515,457,30.165017
2020-12-12,1601,448,27.982511
2020-12-22,974,435,44.661191
2020-12-28,1114,359,32.226212
2020-12-23,775,313,40.387097
2020-12-21,629,275,43.720191
2020-12-02,869,204,23.475259



MONTHLY COVERAGE


,total_observations,valid_targets,coverage_percent
year_month,,,
2020-07,44,12,27.272727
2020-08,7298,3624,49.657440
2020-09,204,1,0.490196
2020-10,13,0,0.000000
2020-11,4762,398,8.357833
2020-12,15345,3681,23.988270
2021-01,2802,118,4.211278
2021-02,1,0,0.000000
2021-03,1743,30,1.721170



TARGET COUNT BY MONTH AND HOUR


hour_local,0,4,5,6,7,8,9,10,11,12,...,14,15,16,17,18,19,20,21,22,23
year_month,,,,,,,,,,,,,,,,,,,,,
2020-07,0,0,0,0,0,0,1,0,0,0,...,0,4,3,0,0,0,1,2,1,0
2020-08,2,0,0,3609,0,1,5,4,0,1,...,0,0,0,0,1,0,0,0,0,0
2020-09,0,0,0,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2020-11,1,0,3,0,0,11,9,3,42,55,...,0,5,2,1,13,21,37,188,0,2
2020-12,0,0,0,34,32,21,53,254,241,194,...,254,114,139,133,136,80,387,1060,347,0
2021-01,0,1,4,0,6,9,8,5,10,11,...,10,4,9,2,7,3,10,5,1,1
2021-03,0,0,0,0,0,0,5,6,11,0,...,0,1,1,2,0,0,0,0,0,0
2021-04,0,0,1,1,2,5,4,3,4,2,...,0,0,0,0,9,2,0,0,0,0



TEMPORAL RANGE
First observation: 2020-07-03 21:30:00+07:00
Last observation: 2021-04-22 13:30:00+07:00
Unique target dates: 75
Unique target timestamps: 378

DAILY TEMPORAL AUDIT COMPLETED!


In [9]:

audit_ts = integrated.copy()

audit_ts["has_target"] = (
    audit_ts["target_los_30m"].notna()
)

# 1. Group by exact observation timestamp
timestamp_audit = (
    audit_ts.groupby("observation_time_local")
    .agg(
        total_rows=("segment_id", "size"),
        unique_segments=("segment_id", "nunique"),
        valid_targets=("has_target", "sum")
    )
    .sort_index()
)

print("TIMESTAMP AUDIT SUMMARY")

print("Unique timestamps:", len(timestamp_audit))
print("Total observations:", timestamp_audit["total_rows"].sum())
print("Total valid targets:", timestamp_audit["valid_targets"].sum())

print("\nObservations per timestamp:")
display(timestamp_audit["total_rows"].describe())

# 2. Top 20 timestamps by number of observations
print("\nTOP 20 TIMESTAMPS BY OBSERVATIONS")

display(
    timestamp_audit.sort_values(
        "total_rows",
        ascending=False
    ).head(20)
)

# 3. Detailed audit for 2020-08-03
aug3_timestamp = timestamp_audit.loc[
    timestamp_audit.index.strftime("%Y-%m-%d") == "2020-08-03"
]

print("\n2020-08-03 TIMESTAMP AUDIT")

print("Number of timestamps:", len(aug3_timestamp))
print("Total observations:", aug3_timestamp["total_rows"].sum())
print("Valid targets:", aug3_timestamp["valid_targets"].sum())

display(
    aug3_timestamp.sort_index()
)

# 4. Distribution of observations per timestamp on August 3
print("\nAUGUST 3 - OBSERVATIONS PER TIMESTAMP")

display(
    aug3_timestamp["total_rows"].describe()
)

# 5. Time intervals between unique timestamps
all_times = pd.Series(
    timestamp_audit.index.sort_values()
)

time_gaps = (
    all_times.diff()
    .dropna()
    .dt.total_seconds() / 60
)

print("\nGLOBAL TIMESTAMP GAP DISTRIBUTION (MINUTES)")

display(time_gaps.describe())

print("\nMOST COMMON TIME GAPS")

display(time_gaps.value_counts().head(15))

# 6. Time intervals within August 3
aug3_times = pd.Series(
    aug3_timestamp.index.sort_values()
)

aug3_gaps = (
    aug3_times.diff()
    .dropna()
    .dt.total_seconds() / 60
)

print("\nAUGUST 3 TIME GAPS (MINUTES)")

display(aug3_gaps.value_counts().head(15))

# 7. Target class distribution by month
valid_targets = audit_ts.loc[
    audit_ts["has_target"]
].copy()

valid_targets["year_month"] = (
    valid_targets["observation_time_local"]
    .dt.strftime("%Y-%m")
)

target_by_month = pd.crosstab(
    valid_targets["year_month"],
    valid_targets["target_los_30m"]
)

print("\nTARGET CLASS DISTRIBUTION BY MONTH")

display(target_by_month)

print("\nTIMESTAMP AUDIT COMPLETED!")


TIMESTAMP AUDIT SUMMARY
Unique timestamps: 827
Total observations: 33441
Total valid targets: 7898

Observations per timestamp:


count     827.000000
mean       40.436518
std       181.958718
min         1.000000
25%         1.000000
50%         4.000000
75%        49.000000
max      3609.000000
Name: total_rows, dtype: float64


TOP 20 TIMESTAMPS BY OBSERVATIONS


,total_rows,unique_segments,valid_targets
observation_time_local,,,
2020-08-03 07:00:00+07:00,3609,3609,0
2020-08-03 06:30:00+07:00,3609,3609,3609
2020-12-24 10:30:00+07:00,295,295,1
2020-12-01 21:00:00+07:00,195,195,74
2020-12-24 07:00:00+07:00,192,192,26
2020-12-01 22:30:00+07:00,187,187,0
2020-12-23 22:00:00+07:00,185,185,48
2020-12-23 21:30:00+07:00,183,183,103
2020-12-23 21:00:00+07:00,183,183,100



2020-08-03 TIMESTAMP AUDIT
Number of timestamps: 5
Total observations: 7223
Valid targets: 3611


,total_rows,unique_segments,valid_targets
observation_time_local,,,
2020-08-03 00:30:00+07:00,2,2,2
2020-08-03 01:00:00+07:00,2,2,0
2020-08-03 06:30:00+07:00,3609,3609,3609
2020-08-03 07:00:00+07:00,3609,3609,0
2020-08-03 08:00:00+07:00,1,1,0


... (7 outputs left out)

observation_time_local
30.0     2
330.0    1
60.0     1
Name: count, dtype: int64


TARGET CLASS DISTRIBUTION BY MONTH


target_los_30m,A,B,C,D,E,F
year_month,,,,,,
2020-07,1,3,2,2,1,3
2020-08,0,27,796,1698,1066,37
2020-09,0,0,0,0,0,1
2020-11,168,115,6,4,7,98
2020-12,2419,1020,27,26,8,181
2021-01,1,2,5,9,15,86
2021-03,0,0,0,0,0,30
2021-04,2,0,1,1,3,27



TIMESTAMP AUDIT COMPLETED!


## 8. Validate exact snapshot matching and observation continuity

The 2020-08-03 snapshot comparison provides a direct check that the generated future LOS label matches the LOS observed 30 minutes later for the same segment.


In [10]:


# PART 1: VERIFY AUGUST 3 SNAPSHOT PAIR

time_0630 = pd.Timestamp(
    "2020-08-03 06:30:00",
    tz="Asia/Ho_Chi_Minh"
)

time_0700 = time_0630 + pd.Timedelta(minutes=30)

# Get two snapshots from original train
snapshot_0630 = train.loc[
    train["observation_time_local"] == time_0630,
    ["segment_id", "LOS"]
].copy()

snapshot_0700 = train.loc[
    train["observation_time_local"] == time_0700,
    ["segment_id", "LOS"]
].copy()

print("AUGUST 3 SNAPSHOT COMPARISON")

print("06:30 observations:", len(snapshot_0630))
print("07:00 observations:", len(snapshot_0700))

# Exact segment matching
snapshot_comparison = snapshot_0630.merge(
    snapshot_0700,
    on="segment_id",
    how="outer",
    suffixes=("_0630", "_0700"),
    indicator=True,
    validate="one_to_one"
)

display(
    snapshot_comparison["_merge"].value_counts()
)

matched_snapshots = snapshot_comparison.loc[
    snapshot_comparison["_merge"] == "both"
].copy()

print(
    "Segments present at both timestamps:",
    len(matched_snapshots)
)

print(
    "Matching segment percentage:",
    round(
        len(matched_snapshots)
        / max(len(snapshot_0630), 1) * 100,
        2
    ),
    "%"
)


# PART 2: VERIFY TARGET LABELS

target_0630 = model_dataset.loc[
    model_dataset["observation_time_local"] == time_0630,
    ["segment_id", "target_los_30m"]
].copy()

target_check = target_0630.merge(
    snapshot_0700.rename(
        columns={"LOS": "LOS_at_0700"}
    ),
    on="segment_id",
    how="left",
    validate="one_to_one"
)

target_check["target_matches_0700"] = (
    target_check["target_los_30m"]
    == target_check["LOS_at_0700"]
)

print("\nTARGET LABEL VALIDATION")

print("Target rows:", len(target_check))

print(
    "Missing 07:00 LOS:",
    target_check["LOS_at_0700"].isna().sum()
)

print(
    "Correct target labels:",
    target_check["target_matches_0700"].sum()
)

print(
    "Incorrect target labels:",
    (~target_check["target_matches_0700"]).sum()
)

assert target_check["LOS_at_0700"].notna().all()
assert target_check["target_matches_0700"].all()


# PART 3: TIME GAP BY SEGMENT

train_times = (
    train[
        ["segment_id", "observation_time_local"]
    ]
    .drop_duplicates()
    .sort_values(
        ["segment_id", "observation_time_local"]
    )
    .copy()
)

train_times["time_gap_minutes"] = (
    train_times.groupby("segment_id")[
        "observation_time_local"
    ]
    .diff()
    .dt.total_seconds() / 60
)

segment_gaps = train_times["time_gap_minutes"].dropna()

print("\nSEGMENT-LEVEL TIME GAP DISTRIBUTION")

display(segment_gaps.describe())

print("\nMOST COMMON SEGMENT TIME GAPS")

display(
    segment_gaps.value_counts().head(15)
)


# PART 4: OBSERVATIONS PER SEGMENT

observations_per_segment = (
    train.groupby("segment_id")
    .size()
)

print("\nOBSERVATIONS PER SEGMENT")

display(observations_per_segment.describe())

print(
    "Segments with only one observation:",
    (observations_per_segment == 1).sum()
)

print(
    "Segments with at least two observations:",
    (observations_per_segment >= 2).sum()
)

print(
    "Segments with at least three observations:",
    (observations_per_segment >= 3).sum()
)

print("\nSNAPSHOT AND CONTINUITY AUDIT COMPLETED!")


AUGUST 3 SNAPSHOT COMPARISON
06:30 observations: 3609
07:00 observations: 3609


_merge
both          3609
left_only        0
right_only       0
Name: count, dtype: int64

Segments present at both timestamps: 3609
Matching segment percentage: 100.0 %

TARGET LABEL VALIDATION
Target rows: 3609
Missing 07:00 LOS: 0
Correct target labels: 3609
Incorrect target labels: 0

SEGMENT-LEVEL TIME GAP DISTRIBUTION


count     23414.000000
mean      16366.854019
std       44150.262166
min          30.000000
25%          30.000000
50%         540.000000
75%        7140.000000
max      381390.000000
Name: time_gap_minutes, dtype: float64


MOST COMMON SEGMENT TIME GAPS


time_gap_minutes
30.0      7898
60.0      1699
90.0       540
1440.0     329
1380.0     318
120.0      309
1350.0     295
1410.0     246
1320.0     242
150.0      214
210.0      212
1470.0     147
1530.0     140
330.0      133
240.0      132
Name: count, dtype: int64


OBSERVATIONS PER SEGMENT


count    10027.000000
mean         3.335095
std          4.698979
min          1.000000
25%          1.000000
50%          2.000000
75%          3.000000
max         54.000000
dtype: float64

Segments with only one observation: 3096
Segments with at least two observations: 6931
Segments with at least three observations: 2876

SNAPSHOT AND CONTINUITY AUDIT COMPLETED!


## 9. Assess chronological split feasibility

Review the distribution of valid target samples over time and compare candidate chronological boundaries. These summaries support the documented temporal holdout decision.


In [11]:

# 1. Work with valid target samples only
split_audit = model_dataset.copy()

split_audit["local_date"] = (
    split_audit["observation_time_local"]
    .dt.strftime("%Y-%m-%d")
)

# 2. Count samples per date
daily_samples = (
    split_audit.groupby("local_date")
    .agg(
        sample_count=("target_los_30m", "size"),
        unique_segments=("segment_id", "nunique"),
        unique_timestamps=("observation_time_local", "nunique")
    )
    .sort_index()
)

daily_samples["cumulative_samples"] = (
    daily_samples["sample_count"].cumsum()
)

daily_samples["cumulative_percent"] = (
    daily_samples["cumulative_samples"]
    / len(split_audit) * 100
)

print("TEMPORAL SPLIT FEASIBILITY")

print("Total samples:", len(split_audit))
print("Unique dates:", len(daily_samples))

print("\nDAILY SAMPLE DISTRIBUTION")

display(
    daily_samples.sort_values(
        "sample_count",
        ascending=False
    ).head(20)
)

# 3. Chronological cumulative distribution
print("\nCHRONOLOGICAL DAILY DISTRIBUTION")

display(daily_samples)

# 4. Examine candidate chronological boundaries
total = len(split_audit)

for ratio in [0.60, 0.70, 0.80]:

    target_count = total * ratio

    boundary = daily_samples[
        daily_samples["cumulative_samples"] >= target_count
    ].index[0]

    print(
        f"\n{ratio:.0%} boundary:",
        boundary
    )

    print(
        "Cumulative samples:",
        daily_samples.loc[
            boundary,
            "cumulative_samples"
        ]
    )

# 5. Evaluate sample distribution by target class
print("\nTARGET CLASS DISTRIBUTION")

display(
    split_audit["target_los_30m"]
    .value_counts()
    .sort_index()
    .to_frame("count")
)

# 6. Evaluate unique dates per month
print("\nUNIQUE TARGET DATES BY MONTH")

split_audit["year_month"] = (
    split_audit["observation_time_local"]
    .dt.strftime("%Y-%m")
)

display(
    split_audit.groupby("year_month")
    .agg(
        samples=("target_los_30m", "size"),
        unique_dates=("local_date", "nunique"),
        unique_segments=("segment_id", "nunique")
    )
)

print("\nTEMPORAL SPLIT AUDIT COMPLETED!")


TEMPORAL SPLIT FEASIBILITY
Total samples: 7898
Unique dates: 75

DAILY SAMPLE DISTRIBUTION


,sample_count,unique_segments,unique_timestamps,cumulative_samples,cumulative_percent
local_date,,,,,
2020-08-03,3611,3611,2,3625,45.897696
2020-12-26,461,139,11,7189,91.023044
2020-12-06,457,206,25,4891,61.927070
2020-12-12,448,197,16,5550,70.270955
2020-12-22,435,203,5,6324,80.070904
2020-12-28,359,182,15,7712,97.644973
2020-12-23,313,155,4,6637,84.033933
2020-12-21,275,156,4,5889,74.563181
2020-12-02,204,137,13,4412,55.862244



CHRONOLOGICAL DAILY DISTRIBUTION


,sample_count,unique_segments,unique_timestamps,cumulative_samples,cumulative_percent
local_date,,,,,
2020-07-04,8,2,8,8,0.101291
2020-07-11,3,1,3,11,0.139276
2020-07-20,1,1,1,12,0.151937
2020-08-02,2,2,2,14,0.177260
2020-08-03,3611,3611,2,3625,45.897696
...,...,...,...,...,...
2021-04-07,1,1,1,7874,99.696126
2021-04-14,1,1,1,7875,99.708787
2021-04-16,3,3,2,7878,99.746771



60% boundary: 2020-12-06
Cumulative samples: 4891

70% boundary: 2020-12-12
Cumulative samples: 5550

80% boundary: 2020-12-22
Cumulative samples: 6324

TARGET CLASS DISTRIBUTION


,count
target_los_30m,
A,2591
B,1167
C,837
D,1740
E,1100
F,463



UNIQUE TARGET DATES BY MONTH


,samples,unique_dates,unique_segments
year_month,,,
2020-07,12,3,3
2020-08,3624,5,3616
2020-09,1,1,1
2020-11,398,18,277
2020-12,3681,27,989
2021-01,118,12,48
2021-03,30,3,17
2021-04,34,6,20



TEMPORAL SPLIT AUDIT COMPLETED!


In [12]:


df_split = model_dataset.copy()

df_split["local_date"] = (
    df_split["observation_time_local"]
    .dt.date
)

df_split = df_split.sort_values(
    ["observation_time_utc", "segment_id"]
)

# Candidate splits based on chronological dates
candidate_splits = {
    "Split_A": ("2020-12-06", "2020-12-22"),
    "Split_B": ("2020-12-12", "2020-12-26"),
    "Split_C": ("2020-12-12", "2020-12-28")
}

los_classes = ["A", "B", "C", "D", "E", "F"]

summary_rows = []
class_rows = []

for split_name, (train_end, val_end) in candidate_splits.items():

    train_end = pd.Timestamp(train_end).date()
    val_end = pd.Timestamp(val_end).date()

    # Each local date belongs to exactly one set.
    train_part = df_split[
        df_split["local_date"] <= train_end
    ]

    val_part = df_split[
        (df_split["local_date"] > train_end)
        & (df_split["local_date"] <= val_end)
    ]

    test_part = df_split[
        df_split["local_date"] > val_end
    ]

    partitions = {
        "Train": train_part,
        "Validation": val_part,
        "Test": test_part
    }

    for set_name, part in partitions.items():

        counts = (
            part["target_los_30m"]
            .value_counts()
            .reindex(los_classes, fill_value=0)
        )

        summary_rows.append({
            "Split": split_name,
            "Set": set_name,
            "Samples": len(part),
            "Unique_Dates": part["local_date"].nunique(),
            "Unique_Segments": part["segment_id"].nunique(),
            "Number_Of_Classes": int((counts > 0).sum()),
            "Missing_Classes": ", ".join(
                counts[counts == 0].index
            ) or "None"
        })

        for los in los_classes:
            class_rows.append({
                "Split": split_name,
                "Set": set_name,
                "LOS": los,
                "Count": int(counts[los])
            })

split_summary = pd.DataFrame(summary_rows)
split_classes = pd.DataFrame(class_rows)

print("TEMPORAL SPLIT SUMMARY")
display(split_summary)

print("\nCLASS DISTRIBUTION")

for split_name in candidate_splits:

    print(f"\n{split_name}")

    pivot = (
        split_classes[
            split_classes["Split"] == split_name
        ]
        .pivot(
            index="Set",
            columns="LOS",
            values="Count"
        )
        .reindex(["Train", "Validation", "Test"])
    )

    display(pivot)

print("\nTEMPORAL SPLIT AUDIT COMPLETED!")


TEMPORAL SPLIT SUMMARY


,Split,Set,Samples,Unique_Dates,Unique_Segments,Number_Of_Classes,Missing_Classes
0,Split_A,Train,4891,32,3986,6,None
1,Split_A,Validation,1433,14,514,6,None
2,Split_A,Test,1574,29,622,6,None
3,Split_B,Train,5550,37,4055,6,None
4,Split_B,Validation,1639,13,486,6,None
5,Split_B,Test,709,25,282,6,None
6,Split_C,Train,5550,37,4055,6,None
7,Split_C,Validation,2162,15,683,6,None
8,Split_C,Test,186,23,80,6,None



CLASS DISTRIBUTION

Split_A


LOS,A,B,C,D,E,F
Set,,,,,,
Train,956,154,805,1705,1076,195
Validation,662,698,10,9,1,53
Test,973,315,22,26,23,215



Split_B


LOS,A,B,C,D,E,F
Set,,,,,,
Train,1597,159,805,1705,1076,208
Validation,475,1006,26,25,6,101
Test,519,2,6,10,18,154



Split_C


LOS,A,B,C,D,E,F
Set,,,,,,
Train,1597,159,805,1705,1076,208
Validation,991,1006,26,25,6,108
Test,3,2,6,10,18,147



TEMPORAL SPLIT AUDIT COMPLETED!


## 10. Create the chronological Train/Holdout partition

Use 2020-12-06 as the final training observation date and 2020-12-07 as the beginning of Holdout. Purge any training row whose target timestamp reaches or crosses the Holdout boundary.


In [13]:

# 1. Copy valid target dataset
final_data = model_dataset.copy()

# 2. Ensure datetime columns
final_data["observation_time_local"] = pd.to_datetime(
    final_data["observation_time_local"]
)

final_data["observation_time_utc"] = pd.to_datetime(
    final_data["observation_time_utc"],
    utc=True
)

# 3. Explicitly create target timestamp
final_data["target_time_utc"] = (
    final_data["observation_time_utc"]
    + pd.Timedelta(minutes=30)
)

# 4. Define chronological boundary
train_end_local = pd.Timestamp(
    "2020-12-06 23:59:59",
    tz="Asia/Ho_Chi_Minh"
)

holdout_start_local = pd.Timestamp(
    "2020-12-07 00:00:00",
    tz="Asia/Ho_Chi_Minh"
)

holdout_start_utc = holdout_start_local.tz_convert("UTC")

# 5. Initial chronological partition
train_candidates = final_data.loc[
    final_data["observation_time_local"]
    <= train_end_local
].copy()

holdout = final_data.loc[
    final_data["observation_time_local"]
    >= holdout_start_local
].copy()

# 6. Purge training samples whose target crosses
#    the holdout boundary
purged_train = train_candidates.loc[
    train_candidates["target_time_utc"]
    >= holdout_start_utc
].copy()

train_final = train_candidates.loc[
    train_candidates["target_time_utc"]
    < holdout_start_utc
].copy()


# 7. Dataset size validation

print("FINAL DATASET SUMMARY")

print("Original valid target samples:", len(final_data))
print("Train before purge:", len(train_candidates))
print("Purged training samples:", len(purged_train))
print("Final train samples:", len(train_final))
print("Final holdout samples:", len(holdout))

print(
    "Total after split:",
    len(train_final) + len(holdout)
)


# 8. Temporal leakage validation

print("\nTEMPORAL LEAKAGE VALIDATION")

print(
    "Latest training observation:",
    train_final["observation_time_local"].max()
)

print(
    "Latest training target:",
    train_final["target_time_utc"].max()
)

print(
    "Earliest holdout observation:",
    holdout["observation_time_local"].min()
)

assert (
    train_final["target_time_utc"].max()
    < holdout_start_utc
), "Temporal leakage detected!"

assert (
    holdout["observation_time_local"].min()
    >= holdout_start_local
), "Holdout contains an earlier observation!"


# 9. Class distribution

classes = ["A", "B", "C", "D", "E", "F"]

class_distribution = pd.DataFrame({
    "Train": train_final["target_los_30m"].value_counts(),
    "Holdout": holdout["target_los_30m"].value_counts()
}).reindex(classes).fillna(0).astype(int)

print("\nCLASS DISTRIBUTION")

display(class_distribution)


# 10. Segment overlap

train_segments = set(train_final["segment_id"])
holdout_segments = set(holdout["segment_id"])

common_segments = train_segments & holdout_segments

print("\nSEGMENT OVERLAP AUDIT")

print("Train unique segments:", len(train_segments))
print("Holdout unique segments:", len(holdout_segments))
print("Common segments:", len(common_segments))

print(
    "Holdout segments previously observed:",
    round(
        len(common_segments) / max(len(holdout_segments), 1) * 100,
        2
    ),
    "%"
)


# 11. Final checks

assert len(train_final) + len(holdout) + len(purged_train) == len(final_data)

assert train_final["target_los_30m"].notna().all()
assert holdout["target_los_30m"].notna().all()

print("\nFINAL TEMPORAL DATASET CREATED!")


FINAL DATASET SUMMARY
Original valid target samples: 7898
Train before purge: 4891
Purged training samples: 0
Final train samples: 4891
Final holdout samples: 3007
Total after split: 7898

TEMPORAL LEAKAGE VALIDATION
Latest training observation: 2020-12-06 21:30:00+07:00
Latest training target: 2020-12-06 15:00:00+00:00
Earliest holdout observation: 2020-12-07 16:00:00+07:00

CLASS DISTRIBUTION


,Train,Holdout
target_los_30m,,
A,956,1635
B,154,1013
C,805,32
D,1705,35
E,1076,24
F,195,268



SEGMENT OVERLAP AUDIT
Train unique segments: 3986
Holdout unique segments: 968
Common segments: 347
Holdout segments previously observed: 35.85 %

FINAL TEMPORAL DATASET CREATED!


## 11. Inspect and refine the feature schema

Inspect candidate feature groups, retain the selected current LOS, temporal, road, and weather attributes, and create cyclical encodings for weekday and month.


In [14]:

# PART 1: FINAL DATASET SCHEMA


print("[FINAL DATASET SCHEMA]")

print("Train shape:", train_final.shape)
print("Holdout shape:", holdout.shape)

print("\nTRAIN COLUMNS AND DATA TYPES")

display(
    train_final.dtypes.to_frame("dtype")
)


# PART 2: FEATURE CANDIDATE INSPECTION

columns = list(train_final.columns)

feature_groups = {
    "IDENTIFIERS": [
        c for c in columns
        if any(x in c.lower() for x in [
            "id", "_id", "name"
        ])
    ],

    "TIME FEATURES": [
        c for c in columns
        if any(x in c.lower() for x in [
            "time", "date", "hour",
            "weekday", "month", "period"
        ])
    ],

    "TRAFFIC FEATURES": [
        c for c in columns
        if any(x in c.lower() for x in [
            "los", "velocity", "speed"
        ])
    ],

    "ROAD FEATURES": [
        c for c in columns
        if any(x in c.lower() for x in [
            "length", "street", "road",
            "longitude", "latitude", "level"
        ])
    ],

    "WEATHER FEATURES": [
        c for c in columns
        if any(x in c.lower() for x in [
            "temperature", "humidity",
            "precipitation", "rain",
            "weather", "wind", "cloud"
        ])
    ]
}

print("\nFEATURE CANDIDATES")

for group, cols in feature_groups.items():

    print(f"\n{group}")

    for col in cols:
        print(" -", col)


# PART 3: TARGET AND CURRENT LOS AUDIT

print("\n[LOS TARGET AUDIT]")

print("Target column exists:",
      "target_los_30m" in train_final.columns)

print("Current LOS exists:",
      "LOS" in train_final.columns)

if "LOS" in train_final.columns:

    print("\nCURRENT LOS DISTRIBUTION")

    display(
        train_final["LOS"].value_counts(
            dropna=False
        ).sort_index()
    )

if "target_los_30m" in train_final.columns:

    print("\nFUTURE LOS DISTRIBUTION")

    display(
        train_final["target_los_30m"]
        .value_counts()
        .sort_index()
    )


# PART 4: WEATHER FEATURE AUDIT

print("\n[WEATHER FEATURE AUDIT]")

weather_cols = [
    c for c in train_final.columns
    if any(x in c.lower() for x in [
        "temperature", "humidity",
        "precipitation", "rain",
        "weather_code", "wind_speed",
        "cloud_cover"
    ])
]

print("Weather columns:", weather_cols)

if "weather_age_minutes" in train_final.columns:

    print("\nWEATHER AGE DISTRIBUTION")

    display(
        train_final["weather_age_minutes"]
        .describe()
    )

# Find weather timestamp columns
weather_time_cols = [
    c for c in train_final.columns
    if "weather" in c.lower()
    and "time" in c.lower()
]

print("\nWeather timestamp columns:",
      weather_time_cols)

for col in weather_time_cols:

    print(f"\n{col}")

    display(train_final[col].head(5))


# PART 5: SEGMENT STATUS SOURCE AUDIT


print("\n[SEGMENT STATUS SOURCE AUDIT]")

status_candidates = [
    "segment_status",
    "status",
    "segment_status_preprocessed"
]

for name in status_candidates:

    obj = globals().get(name)

    if isinstance(obj, pd.DataFrame):

        print(f"\nFound DataFrame: {name}")

        print("Shape:", obj.shape)

        print("Columns:", list(obj.columns))

        if "updated_at" in obj.columns:

            print("\nUpdated_at sample:")

            display(
                obj["updated_at"].head(10)
            )

            print(
                "Updated_at dtype:",
                obj["updated_at"].dtype
            )

        if "velocity" in obj.columns:

            print("\nVelocity statistics:")

            display(
                obj["velocity"].describe()
            )


# PART 6: FINAL MISSING VALUE AUDIT

print("\n[MISSING VALUE AUDIT]")

missing_report = pd.DataFrame({
    "Missing_Count": train_final.isna().sum(),
    "Missing_Percent": (
        train_final.isna().mean() * 100
    )
})

display(
    missing_report.sort_values(
        "Missing_Percent",
        ascending=False
    ).head(30)
)

print("\nFEATURE AVAILABILITY AUDIT COMPLETED!")


[FINAL DATASET SCHEMA]
Train shape: (4891, 51)
Holdout shape: (3007, 51)

TRAIN COLUMNS AND DATA TYPES


,dtype
_id,int64
segment_id,int64
date,str
weekday,int64
period,str
LOS,str
s_node_id_x,int64
e_node_id_x,int64
length_x,int64
street_id,int64



FEATURE CANDIDATES

IDENTIFIERS
 - _id
 - segment_id
 - s_node_id_x
 - e_node_id_x
 - street_id
 - street_name
 - s_node_id_y
 - e_node_id_y
 - relative_humidity_2m

TIME FEATURES
 - date
 - weekday
 - period
 - hour_utc
 - observation_time_utc
 - target_time_utc
 - observation_time_local
 - hour_local
 - weekday_local
 - month_local
 - hour_sin
 - hour_cos
 - weather_time_utc

TRAFFIC FEATURES
 - LOS
 - max_velocity
 - wind_speed_10m
 - target_los_30m

ROAD FEATURES
 - length_x
 - street_id
 - street_level_x
 - street_name
 - street_type_x
 - length_y
 - street_level_y
 - street_type_y
 - start_longitude
 - start_latitude
 - end_longitude
 - end_latitude
 - center_longitude
 - center_latitude
 - length_zero_flag

WEATHER FEATURES
 - weather_time_utc
 - temperature_2m
 - relative_humidity_2m
 - precipitation
 - weather_code
 - wind_speed_10m
 - cloud_cover
 - weather_age_minutes

[LOS TARGET AUDIT]
Target column exists: True
Current LOS exists: True

CURRENT LOS DISTRIBUTION


LOS
A    1125
B     312
C    1115
D     474
E    1193
F     672
Name: count, dtype: int64


FUTURE LOS DISTRIBUTION


target_los_30m
A     956
B     154
C     805
D    1705
E    1076
F     195
Name: count, dtype: int64

... (7 outputs left out)

count    90938.000000
mean        20.754844
std         19.567112
min          0.000000
25%          2.000000
50%         17.000000
75%         34.000000
max        200.000000
Name: velocity, dtype: float64


[MISSING VALUE AUDIT]


,Missing_Count,Missing_Percent
max_velocity,4380,89.552239
segment_id,0,0.000000
_id,0,0.000000
date,0,0.000000
weekday,0,0.000000
LOS,0,0.000000
period,0,0.000000
s_node_id_x,0,0.000000
e_node_id_x,0,0.000000
length_x,0,0.000000



FEATURE AVAILABILITY AUDIT COMPLETED!


In [15]:

# 1. Define feature columns
FEATURE_COLUMNS = [
    "LOS",

    # Time
    "hour_sin",
    "hour_cos",
    "weekday_local",
    "month_local",
    "is_weekend",
    "minute_local",

    # Road
    "length_y",
    "street_level_y",
    "street_type_y",
    "center_longitude",
    "center_latitude",

    # Weather
    "temperature_2m",
    "relative_humidity_2m",
    "precipitation",
    "weather_code",
    "wind_speed_10m",
    "cloud_cover"
]

TARGET_COLUMN = "target_los_30m"

# 2. Validate required columns
missing_train = [
    c for c in FEATURE_COLUMNS + [TARGET_COLUMN]
    if c not in train_final.columns
]

missing_holdout = [
    c for c in FEATURE_COLUMNS + [TARGET_COLUMN]
    if c not in holdout.columns
]

assert not missing_train, f"Missing Train columns: {missing_train}"
assert not missing_holdout, f"Missing Holdout columns: {missing_holdout}"

# 3. Create clean datasets
train_clean = train_final[
    FEATURE_COLUMNS + [TARGET_COLUMN]
].copy()

holdout_clean = holdout[
    FEATURE_COLUMNS + [TARGET_COLUMN]
].copy()

# 4. Rename duplicated road columns
rename_map = {
    "length_y": "length",
    "street_level_y": "street_level",
    "street_type_y": "street_type"
}

train_clean = train_clean.rename(columns=rename_map)
holdout_clean = holdout_clean.rename(columns=rename_map)

# 5. Validate target
assert train_clean[TARGET_COLUMN].notna().all()
assert holdout_clean[TARGET_COLUMN].notna().all()

# 6. Check missing values
print("TRAIN CLEAN DATASET")
print("Shape:", train_clean.shape)

display(
    train_clean.isna().sum()
    .to_frame("Missing_Count")
)

print("\nHOLDOUT CLEAN DATASET")
print("Shape:", holdout_clean.shape)

display(
    holdout_clean.isna().sum()
    .to_frame("Missing_Count")
)

# 7. Check feature data types
print("\nFEATURE DATA TYPES")

display(
    train_clean.dtypes.to_frame("dtype")
)

# 8. Check categorical distributions
categorical_features = [
    "LOS",
    "street_type",
    "weather_code"
]

print("\nCATEGORICAL FEATURE DISTRIBUTIONS")

for col in categorical_features:

    print(f"\n{col}")

    display(
        train_clean[col].value_counts(
            dropna=False
        ).head(20)
    )

# 9. Check numerical statistics
numeric_features = [
    "hour_sin",
    "hour_cos",
    "weekday_local",
    "month_local",
    "is_weekend",
    "minute_local",
    "length",
    "street_level",
    "center_longitude",
    "center_latitude",
    "temperature_2m",
    "relative_humidity_2m",
    "precipitation",
    "wind_speed_10m",
    "cloud_cover"
]

print("\nNUMERICAL FEATURE SUMMARY")

display(
    train_clean[numeric_features].describe().T
)

# 10. Final validation
assert list(train_clean.columns) == list(holdout_clean.columns)

print("\nFINAL FEATURE SCHEMA")

print("Features:", len(FEATURE_COLUMNS))
print("Target:", TARGET_COLUMN)
print("Train:", train_clean.shape)
print("Holdout:", holdout_clean.shape)

print("\nFEATURE SCHEMA CLEANING COMPLETED!")


TRAIN CLEAN DATASET
Shape: (4891, 19)


,Missing_Count
LOS,0
hour_sin,0
hour_cos,0
weekday_local,0
month_local,0
is_weekend,0
minute_local,0
length,0
street_level,0
street_type,0



HOLDOUT CLEAN DATASET
Shape: (3007, 19)


,Missing_Count
LOS,0
hour_sin,0
hour_cos,0
weekday_local,0
month_local,0
is_weekend,0
minute_local,0
length,0
street_level,0
street_type,0



FEATURE DATA TYPES


,dtype
LOS,str
hour_sin,float64
hour_cos,float64
weekday_local,int32
month_local,int32
is_weekend,int8
minute_local,int32
length,int64
street_level,int64
street_type,string


... (5 outputs left out)

weather_code
3     4066
51     487
1      157
2      133
53      24
0       16
63       5
61       2
55       1
Name: count, dtype: int64


NUMERICAL FEATURE SUMMARY


,count,mean,std,min,25%,50%,75%,max
hour_sin,4891.0,0.618182,0.676234,-1.000000,0.608761,0.991445,0.991445,1.000000
hour_cos,4891.0,-0.053532,0.397265,-1.000000,-0.130526,-0.130526,-0.130526,1.000000
weekday_local,4891.0,0.956860,2.020544,0.000000,0.000000,0.000000,0.000000,6.000000
month_local,4891.0,8.941934,1.627443,7.000000,8.000000,8.000000,11.000000,12.000000
is_weekend,4891.0,0.130648,0.337050,0.000000,0.000000,0.000000,0.000000,1.000000
minute_local,4891.0,26.178696,10.002861,0.000000,30.000000,30.000000,30.000000,30.000000
length,4891.0,42.866694,45.243895,0.000000,14.000000,29.000000,56.000000,489.000000
street_level,4891.0,3.282560,0.953682,1.000000,2.000000,4.000000,4.000000,4.000000
center_longitude,4891.0,106.654385,0.021594,106.538165,106.646640,106.656780,106.666262,106.805522
center_latitude,4891.0,10.783163,0.035913,10.708826,10.760550,10.769029,10.783736,10.888485



FINAL FEATURE SCHEMA
Features: 18
Target: target_los_30m
Train: (4891, 19)
Holdout: (3007, 19)

FEATURE SCHEMA CLEANING COMPLETED!


In [16]:

# 1. Copy clean datasets
train_fe = train_clean.copy()
holdout_fe = holdout_clean.copy()


# PART 1: CYCLICAL TIME FEATURES

def add_cyclical_features(df):

    df = df.copy()

    # Weekday cyclical encoding
    weekday = df["weekday_local"]

    df["weekday_sin"] = np.sin(
        2 * np.pi * weekday / 7
    )

    df["weekday_cos"] = np.cos(
        2 * np.pi * weekday / 7
    )

    # Month cyclical encoding
    month = df["month_local"]

    df["month_sin"] = np.sin(
        2 * np.pi * (month - 1) / 12
    )

    df["month_cos"] = np.cos(
        2 * np.pi * (month - 1) / 12
    )

    # Remove redundant/original temporal columns
    df = df.drop(
        columns=[
            "weekday_local",
            "month_local",
            "minute_local"
        ]
    )

    return df


train_fe = add_cyclical_features(train_fe)
holdout_fe = add_cyclical_features(holdout_fe)


# PART 2: STREET TYPE NORMALIZATION

non_road_tags = [
    "house",
    "school",
    "company",
    "car",
    "marketplace"
]

for df in [train_fe, holdout_fe]:

    df["street_type"] = (
        df["street_type"]
        .astype("string")
        .str.strip()
        .str.lower()
    )

    df["street_type"] = df["street_type"].replace(
        non_road_tags,
        "unclassified"
    )


# PART 3: CATEGORICAL DATA TYPES

categorical_features = [
    "LOS",
    "street_level",
    "street_type",
    "weather_code"
]

for df in [train_fe, holdout_fe]:

    for col in categorical_features:

        df[col] = df[col].astype("string")


# PART 4: FEATURE LIST

NUMERIC_FEATURES = [
    "hour_sin",
    "hour_cos",

    "weekday_sin",
    "weekday_cos",

    "month_sin",
    "month_cos",

    "is_weekend",

    "length",

    "center_longitude",
    "center_latitude",

    "temperature_2m",
    "relative_humidity_2m",
    "precipitation",
    "wind_speed_10m",
    "cloud_cover"
]

CATEGORICAL_FEATURES = [
    "LOS",
    "street_level",
    "street_type",
    "weather_code"
]

TARGET_COLUMN = "target_los_30m"

FEATURE_COLUMNS = (
    NUMERIC_FEATURES
    + CATEGORICAL_FEATURES
)


# PART 5: DATASET VALIDATION

assert len(FEATURE_COLUMNS) == len(set(FEATURE_COLUMNS))

assert TARGET_COLUMN not in FEATURE_COLUMNS

assert all(
    c in train_fe.columns
    for c in FEATURE_COLUMNS + [TARGET_COLUMN]
)

assert all(
    c in holdout_fe.columns
    for c in FEATURE_COLUMNS + [TARGET_COLUMN]
)

assert list(train_fe[FEATURE_COLUMNS].columns) == list(
    holdout_fe[FEATURE_COLUMNS].columns
)

assert train_fe[FEATURE_COLUMNS].isna().sum().sum() == 0
assert holdout_fe[FEATURE_COLUMNS].isna().sum().sum() == 0


# PART 6: FEATURE SUMMARY

print("FEATURE REFINEMENT RESULTS")

print("Train shape:", train_fe.shape)
print("Holdout shape:", holdout_fe.shape)

print("\nNumerical features:", len(NUMERIC_FEATURES))
print("Categorical features:", len(CATEGORICAL_FEATURES))
print("Total features:", len(FEATURE_COLUMNS))

print("\nNORMALIZED STREET TYPES")

display(
    train_fe["street_type"].value_counts()
)

print("\nTARGET DISTRIBUTION - TRAIN")

display(
    train_fe[TARGET_COLUMN].value_counts()
    .sort_index()
)

print("\nTARGET DISTRIBUTION - HOLDOUT")

display(
    holdout_fe[TARGET_COLUMN].value_counts()
    .sort_index()
)

print("\nFINAL FEATURE LIST")

for i, col in enumerate(FEATURE_COLUMNS, 1):
    print(f"{i:02d}. {col}")

print("\nFEATURE REFINEMENT COMPLETED!")


FEATURE REFINEMENT RESULTS
Train shape: (4891, 20)
Holdout shape: (3007, 20)

Numerical features: 15
Categorical features: 4
Total features: 19

NORMALIZED STREET TYPES


street_type
tertiary          2563
primary           1167
secondary          525
unclassified       331
trunk              187
primary_link        51
secondary_link      29
tertiary_link       16
residential         13
trunk_link           9
Name: count, dtype: int64[pyarrow]


TARGET DISTRIBUTION - TRAIN


target_los_30m
A     956
B     154
C     805
D    1705
E    1076
F     195
Name: count, dtype: int64


TARGET DISTRIBUTION - HOLDOUT


target_los_30m
A    1635
B    1013
C      32
D      35
E      24
F     268
Name: count, dtype: int64


FINAL FEATURE LIST
01. hour_sin
02. hour_cos
03. weekday_sin
04. weekday_cos
05. month_sin
06. month_cos
07. is_weekend
08. length
09. center_longitude
10. center_latitude
11. temperature_2m
12. relative_humidity_2m
13. precipitation
14. wind_speed_10m
15. cloud_cover
16. LOS
17. street_level
18. street_type
19. weather_code

FEATURE REFINEMENT COMPLETED!


## 12. Audit category availability and Train/Holdout feature drift

Compare categorical levels and numerical summaries between Train and Holdout. This is a diagnostic audit: the chronological Holdout distribution is preserved and is not modified to resemble Train.


In [17]:

# PART 1: UNSEEN CATEGORIES


print("[UNSEEN CATEGORY AUDIT]")

category_results = []

for col in CATEGORICAL_FEATURES:

    train_values = set(
        train_fe[col].dropna().astype(str)
    )

    holdout_values = set(
        holdout_fe[col].dropna().astype(str)
    )

    unseen = holdout_values - train_values

    unseen_count = holdout_fe[col].astype(str).isin(
        unseen
    ).sum()

    category_results.append({
        "Feature": col,
        "Train_Categories": len(train_values),
        "Holdout_Categories": len(holdout_values),
        "Unseen_Categories": len(unseen),
        "Unseen_Values": ", ".join(sorted(unseen)) or "None",
        "Affected_Rows": int(unseen_count),
        "Affected_Percent": round(
            unseen_count / len(holdout_fe) * 100, 2
        )
    })

category_audit = pd.DataFrame(category_results)

display(category_audit)


# PART 2: CATEGORICAL DISTRIBUTION SHIFT

print("\n[CATEGORICAL DISTRIBUTION SHIFT]")

for col in CATEGORICAL_FEATURES:

    train_dist = (
        train_fe[col]
        .value_counts(normalize=True)
        .rename("Train_Percent")
    )

    holdout_dist = (
        holdout_fe[col]
        .value_counts(normalize=True)
        .rename("Holdout_Percent")
    )

    comparison = pd.concat(
        [train_dist, holdout_dist],
        axis=1
    ).fillna(0) * 100

    comparison["Absolute_Difference"] = (
        comparison["Train_Percent"]
        - comparison["Holdout_Percent"]
    ).abs()

    print(f"\n{col}")

    display(
        comparison.sort_values(
            "Absolute_Difference",
            ascending=False
        )
    )


# PART 3: NUMERICAL FEATURE DRIFT


print("\n[NUMERICAL FEATURE DRIFT]")

numeric_results = []

for col in NUMERIC_FEATURES:

    train_mean = train_fe[col].mean()
    holdout_mean = holdout_fe[col].mean()

    train_std = train_fe[col].std()
    holdout_std = holdout_fe[col].std()

    # Standardized mean difference
    pooled_std = np.sqrt(
        (train_std**2 + holdout_std**2) / 2
    )

    smd = (
        (holdout_mean - train_mean) / pooled_std
        if pooled_std > 0 else 0
    )

    numeric_results.append({
        "Feature": col,
        "Train_Mean": train_mean,
        "Holdout_Mean": holdout_mean,
        "Train_Std": train_std,
        "Holdout_Std": holdout_std,
        "SMD": smd
    })

numeric_drift = pd.DataFrame(numeric_results)

numeric_drift["Abs_SMD"] = (
    numeric_drift["SMD"].abs()
)

display(
    numeric_drift.sort_values(
        "Abs_SMD",
        ascending=False
    ).round(4)
)


# PART 4: TARGET VALIDATION

print("\n[TARGET CLASS VALIDATION]")

train_classes = set(
    train_fe[TARGET_COLUMN].unique()
)

holdout_classes = set(
    holdout_fe[TARGET_COLUMN].unique()
)

print("Train classes:", sorted(train_classes))
print("Holdout classes:", sorted(holdout_classes))

print(
    "Missing target classes:",
    sorted(train_classes - holdout_classes)
)

print("\nCATEGORY & FEATURE DRIFT AUDIT COMPLETED!")


[UNSEEN CATEGORY AUDIT]


,Feature,Train_Categories,Holdout_Categories,Unseen_Categories,Unseen_Values,Affected_Rows,Affected_Percent
0,LOS,6,6,0,None,0,0.0
1,street_level,4,4,0,None,0,0.0
2,street_type,10,9,0,None,0,0.0
3,weather_code,9,8,0,None,0,0.0



[CATEGORICAL DISTRIBUTION SHIFT]

LOS


,Train_Percent,Holdout_Percent,Absolute_Difference
LOS,,,
A,23.001431,54.439641,31.43821
B,6.379064,33.85434,27.475276
E,24.39174,0.831393,23.560347
C,22.796974,0.864649,21.932325
D,9.69127,0.931161,8.760109
F,13.739522,9.078816,4.660705



street_level


,Train_Percent,Holdout_Percent,Absolute_Difference
street_level,,,
2,23.860151,49.750582,25.890431
4,59.76283,38.676422,21.086408
3,12.55367,7.482541,5.071129
1,3.823349,4.090456,0.267107


... (3 outputs left out)

,Train_Percent,Holdout_Percent,Absolute_Difference
weather_code,,,
3,83.132284,48.187562,34.944721
51,9.957064,33.189225,23.232161
2,2.71928,10.342534,7.623254
61,0.040891,1.79581,1.754918
1,3.209978,4.489524,1.279547
0,0.327131,1.263718,0.936587
53,0.490697,0.69837,0.207673
63,0.102229,0.033256,0.068973
55,0.020446,0.0,0.020446



[NUMERICAL FEATURE DRIFT]


,Feature,Train_Mean,Holdout_Mean,Train_Std,Holdout_Std,SMD,Abs_SMD
5,month_cos,-0.4520,0.8578,0.7111,0.1023,2.5785,2.5785
11,relative_humidity_2m,89.6984,67.0329,10.1664,12.7816,-1.9627,1.9627
0,hour_sin,0.6182,-0.4346,0.6762,0.4823,-1.7925,1.7925
10,temperature_2m,25.8004,28.3028,1.4322,2.7261,1.1492,1.1492
3,weekday_cos,0.8394,0.2676,0.3800,0.6043,-1.1329,1.1329
9,center_latitude,10.7832,10.8103,0.0359,0.0338,0.7783,0.7783
6,is_weekend,0.1306,0.3725,0.3370,0.4835,0.5802,0.5802
4,month_sin,-0.5286,-0.4498,0.1036,0.2266,0.4475,0.4475
14,cloud_cover,90.8538,82.5052,15.2113,22.6101,-0.4333,0.4333
8,center_longitude,106.6544,106.6645,0.0216,0.0311,0.3789,0.3789



[TARGET CLASS VALIDATION]
Train classes: ['A', 'B', 'C', 'D', 'E', 'F']
Holdout classes: ['A', 'B', 'C', 'D', 'E', 'F']
Missing target classes: []

CATEGORY & FEATURE DRIFT AUDIT COMPLETED!


## 13. Validate the in-memory V2 candidates before road-length filtering

This replaces the former V1 export/reload workflow. The V2 feature tables and their metadata are prepared directly from the current in-memory Train/Holdout partitions, so this notebook no longer depends on `model_v1` files.

The feature rows and metadata rows are reset in the same order. Metadata is kept separately and is not included as model features.


In [18]:

# PRE-FILTER V2 FEATURE AND METADATA AUDIT


TARGET_COLUMN = "target_los_30m"
LOS_CLASSES = ["A", "B", "C", "D", "E", "F"]

METADATA_COLUMNS = [
    "segment_id",
    "observation_time_local",
    "observation_time_utc",
    "target_time_utc"
]

# Preserve the same row order as the feature tables.
train_fe = train_fe.reset_index(drop=True).copy()
holdout_fe = holdout_fe.reset_index(drop=True).copy()

train_meta_prefilter = (
    train_final[METADATA_COLUMNS]
    .reset_index(drop=True)
    .copy()
)

holdout_meta_prefilter = (
    holdout[METADATA_COLUMNS]
    .reset_index(drop=True)
    .copy()
)

# Keep source counts for the final report.
SOURCE_OBSERVATION_COUNT = len(train)
VALID_TARGET_COUNT_BEFORE_ROAD_FILTER = len(model_dataset)
TARGET_COVERAGE_PERCENT = (
    VALID_TARGET_COUNT_BEFORE_ROAD_FILTER
    / SOURCE_OBSERVATION_COUNT
    * 100
)

# 1. Feature/metadata row alignment
assert len(train_fe) == len(train_meta_prefilter)
assert len(holdout_fe) == len(holdout_meta_prefilter)

assert train_fe.index.equals(train_meta_prefilter.index)
assert holdout_fe.index.equals(holdout_meta_prefilter.index)

# 2. Feature schema consistency
assert list(train_fe.columns) == list(holdout_fe.columns)
assert TARGET_COLUMN in train_fe.columns

metadata_or_identifier_columns = {
    "segment_id",
    "observation_time_local",
    "observation_time_utc",
    "target_time_utc",
    "sample_id",
    "row_order",
    "_id"
}

assert not (
    set(train_fe.columns) & metadata_or_identifier_columns
), "Identifier or timestamp metadata leaked into feature columns."

# 3. Missing and infinite values
assert train_fe.isna().sum().sum() == 0
assert holdout_fe.isna().sum().sum() == 0

assert np.isfinite(
    train_fe[NUMERIC_FEATURES].to_numpy(dtype=float)
).all()

assert np.isfinite(
    holdout_fe[NUMERIC_FEATURES].to_numpy(dtype=float)
).all()

# 4. Target and sample-key validation
assert set(train_fe[TARGET_COLUMN].unique()) == set(LOS_CLASSES)
assert set(holdout_fe[TARGET_COLUMN].unique()) == set(LOS_CLASSES)

sample_key_columns = ["segment_id", "observation_time_utc"]

assert not train_meta_prefilter.duplicated(sample_key_columns).any()
assert not holdout_meta_prefilter.duplicated(sample_key_columns).any()

# 5. Exact 30-minute target interval
for name, metadata in [
    ("Train", train_meta_prefilter),
    ("Holdout", holdout_meta_prefilter)
]:
    observation_time = pd.to_datetime(
        metadata["observation_time_utc"], utc=True
    )
    target_time = pd.to_datetime(
        metadata["target_time_utc"], utc=True
    )

    interval_minutes = (
        target_time - observation_time
    ).dt.total_seconds() / 60

    assert interval_minutes.eq(30).all()

    print(
        f"{name}: {len(metadata):,} rows; "
        f"target interval = {sorted(interval_minutes.unique())} minutes"
    )

print("\nPRE-FILTER V2 CANDIDATE AUDIT")
print("Original observations:", SOURCE_OBSERVATION_COUNT)
print("Valid future targets:", VALID_TARGET_COUNT_BEFORE_ROAD_FILTER)
print(f"Target coverage: {TARGET_COVERAGE_PERCENT:.2f}%")
print("Train candidate:", train_fe.shape)
print("Holdout candidate:", holdout_fe.shape)
print("Train metadata:", train_meta_prefilter.shape)
print("Holdout metadata:", holdout_meta_prefilter.shape)
print("\nPRE-FILTER V2 CANDIDATE AUDIT PASSED!")


Train: 4,891 rows; target interval = [np.float64(30.0)] minutes
Holdout: 3,007 rows; target interval = [np.float64(30.0)] minutes

PRE-FILTER V2 CANDIDATE AUDIT
Original observations: 33441
Valid future targets: 7898
Target coverage: 23.62%
Train candidate: (4891, 20)
Holdout candidate: (3007, 20)
Train metadata: (4891, 4)
Holdout metadata: (3007, 4)

PRE-FILTER V2 CANDIDATE AUDIT PASSED!


## 14. Audit source road-length precision

The source `length` field is stored as an integer. Compare it with the Haversine distance between the segment's endpoint coordinates.

The purpose of this audit is to understand why some source lengths are zero. **Do not replace the source length with Haversine distance in this pipeline.** The geometric calculation is used only as a quality cross-check.


In [19]:

# SOURCE ROAD-LENGTH PRECISION AUDIT

source_length = pd.to_numeric(road["length"], errors="coerce")

coordinate_columns = [
    "start_latitude",
    "start_longitude",
    "end_latitude",
    "end_longitude"
]

coordinates = road[coordinate_columns].apply(
    pd.to_numeric, errors="coerce"
)

assert source_length.notna().all()
assert source_length.ge(0).all()
assert coordinates.notna().all().all()

lat1 = np.radians(coordinates["start_latitude"])
lon1 = np.radians(coordinates["start_longitude"])
lat2 = np.radians(coordinates["end_latitude"])
lon2 = np.radians(coordinates["end_longitude"])

dlat = lat2 - lat1
dlon = lon2 - lon1

a = (
    np.sin(dlat / 2) ** 2
    + np.cos(lat1) * np.cos(lat2)
    * np.sin(dlon / 2) ** 2
)
a = np.clip(a, 0, 1)

EARTH_RADIUS_METERS = 6_371_000

haversine_distance = pd.Series(
    2 * EARTH_RADIUS_METERS * np.arcsin(np.sqrt(a)),
    index=road.index,
    name="haversine_distance_m"
)

# The source integer length matches the floored endpoint distance
# in the supplied data audit.
floor_distance = np.floor(haversine_distance + 1e-9)

floor_match = np.isclose(
    source_length.to_numpy(dtype=float),
    floor_distance.to_numpy(dtype=float)
)

zero_source_length = source_length.eq(0)
sub_meter_by_coordinates = haversine_distance.lt(1)

road_length_audit = pd.DataFrame({
    "segment_id": road["segment_id"],
    "source_length_m": source_length,
    "haversine_distance_m": haversine_distance,
    "floor_haversine_m": floor_distance,
    "floor_match": floor_match,
    "source_length_is_zero": zero_source_length
})


print("[SOURCE ROAD-LENGTH PRECISION AUDIT]")


print("Total segments:", len(road_length_audit))
print("Source length = 0 m:", int(zero_source_length.sum()))
print(
    "Haversine distance < 1 m:",
    int(sub_meter_by_coordinates.sum())
)
print("Source length matches floor(Haversine):", int(floor_match.sum()))
print("Mismatches:", int((~floor_match).sum()))
print(
    "Matching percentage:",
    round(float(floor_match.mean() * 100), 4), "%"
)

print("\nHAVERSINE DISTANCE SUMMARY (METERS)")
display(haversine_distance.describe())

print("\nSOURCE-ZERO-LENGTH EXAMPLES")
display(
    road_length_audit.loc[zero_source_length].head(20)
)

# This confirms the observed source representation; it does not
# establish that every sub-meter graph segment is physically invalid.
assert floor_match.all()
assert (
    zero_source_length == floor_distance.eq(0)
).all()

print("\nSOURCE LENGTH PRECISION AUDIT PASSED!")


[SOURCE ROAD-LENGTH PRECISION AUDIT]
Total segments: 84633
Source length = 0 m: 154
Haversine distance < 1 m: 154
Source length matches floor(Haversine): 84633
Mismatches: 0
Matching percentage: 100.0 %

HAVERSINE DISTANCE SUMMARY (METERS)


count    84633.000000
mean        42.318368
std         52.692198
min          0.010920
25%         13.512458
50%         27.615171
75%         52.130408
max       1622.537117
Name: haversine_distance_m, dtype: float64


SOURCE-ZERO-LENGTH EXAMPLES


,segment_id,source_length_m,haversine_distance_m,floor_haversine_m,floor_match,source_length_is_zero
864,864,0,0.470803,0.0,True,True
865,865,0,0.470803,0.0,True,True
2513,2513,0,0.412946,0.0,True,True
2514,2514,0,0.412946,0.0,True,True
6655,6655,0,0.524465,0.0,True,True
6656,6656,0,0.524465,0.0,True,True
8411,8411,0,0.159083,0.0,True,True
8412,8412,0,0.159083,0.0,True,True
10397,10397,0,0.578815,0.0,True,True
10398,10398,0,0.578815,0.0,True,True



SOURCE LENGTH PRECISION AUDIT PASSED!


## 15. Apply the documented source-length filter

The audit shows that the source integer length equals the floored Haversine endpoint distance for the supplied road table. Therefore, a source value of `0 m` corresponds to an endpoint distance below `1 m` in this dataset.

These segments are described as **source-zero-length / sub-meter segments**, not automatically as malformed roads. The V2 cleaning policy excludes observations mapped to these segments so the final feature table does not contain a zero-valued road-length feature. The source length itself is not overwritten.


In [20]:

# SOURCE-ZERO-LENGTH SEGMENT AUDIT AND FILTER

source_zero_segments = road.loc[
    pd.to_numeric(road["length"], errors="coerce").eq(0),
    ["segment_id", "length"]
].copy()

source_zero_segment_ids = set(
    source_zero_segments["segment_id"]
)

print("SOURCE ROAD-LENGTH FILTER")
print("Source-zero-length segments:", len(source_zero_segment_ids))
print("Source segments with positive length:",
      int(pd.to_numeric(road["length"]).gt(0).sum()))

print("\nExamples:")
display(source_zero_segments.head(10))

# Identify observations associated with source-zero-length segments.
train_keep = (
    ~train_meta_prefilter["segment_id"]
    .isin(source_zero_segment_ids)
    .to_numpy()
)

holdout_keep = (
    ~holdout_meta_prefilter["segment_id"]
    .isin(source_zero_segment_ids)
    .to_numpy()
)

# Preserve excluded records for traceability.
excluded_train = train_meta_prefilter.iloc[
    np.flatnonzero(~train_keep)
].copy()
excluded_train["split"] = "train"

excluded_holdout = holdout_meta_prefilter.iloc[
    np.flatnonzero(~holdout_keep)
].copy()
excluded_holdout["split"] = "holdout"

excluded_samples = pd.concat(
    [excluded_train, excluded_holdout],
    ignore_index=True
)

# Apply the same positional masks to features and metadata.
train_filtered = train_fe.iloc[
    np.flatnonzero(train_keep)
].reset_index(drop=True)

train_meta_filtered = train_meta_prefilter.iloc[
    np.flatnonzero(train_keep)
].reset_index(drop=True)

holdout_filtered = holdout_fe.iloc[
    np.flatnonzero(holdout_keep)
].reset_index(drop=True)

holdout_meta_filtered = holdout_meta_prefilter.iloc[
    np.flatnonzero(holdout_keep)
].reset_index(drop=True)

# Dynamic validation: feature and metadata rows remain aligned.
assert len(train_filtered) == len(train_meta_filtered)
assert len(holdout_filtered) == len(holdout_meta_filtered)

assert len(train_filtered) == int(train_keep.sum())
assert len(holdout_filtered) == int(holdout_keep.sum())

assert train_filtered["length"].gt(0).all()
assert holdout_filtered["length"].gt(0).all()

assert not train_meta_filtered["segment_id"].isin(
    source_zero_segment_ids
).any()

assert not holdout_meta_filtered["segment_id"].isin(
    source_zero_segment_ids
).any()

assert len(excluded_samples) == (
    len(excluded_train) + len(excluded_holdout)
)

print("\nFILTERING RESULTS")
print("Train before:", len(train_fe))
print("Train after:", len(train_filtered))
print("Train removed:", len(excluded_train))

print("\nHoldout before:", len(holdout_fe))
print("Holdout after:", len(holdout_filtered))
print("Holdout removed:", len(excluded_holdout))

print("\nTotal removed:", len(excluded_samples))
print("Final Train shape:", train_filtered.shape)
print("Final Holdout shape:", holdout_filtered.shape)

print("\nSOURCE-LENGTH FILTER VALIDATION PASSED!")


SOURCE ROAD-LENGTH FILTER
Source-zero-length segments: 154
Source segments with positive length: 84479

Examples:


,segment_id,length
864,864,0
865,865,0
2513,2513,0
2514,2514,0
6655,6655,0
6656,6656,0
8411,8411,0
8412,8412,0
10397,10397,0
10398,10398,0



FILTERING RESULTS
Train before: 4891
Train after: 4879
Train removed: 12

Holdout before: 3007
Holdout after: 2998
Holdout removed: 9

Total removed: 21
Final Train shape: (4879, 20)
Final Holdout shape: (2998, 20)

SOURCE-LENGTH FILTER VALIDATION PASSED!


## 16. Validate the filtered datasets

Check missing values, feature/metadata row counts, positive road lengths, unique sample keys, target intervals, Train/Holdout overlap, and chronological separation.


In [21]:

# FINAL DATASET INTEGRITY AUDIT

datasets = {
    "Train": (train_filtered, train_meta_filtered),
    "Holdout": (holdout_filtered, holdout_meta_filtered)
}

for name, (df, meta) in datasets.items():


    print(f"\n[{name.upper()} INTEGRITY AUDIT]\n")


    # 1. Shape validation
    print("Feature shape:", df.shape)
    print("Metadata shape:", meta.shape)

    assert len(df) == len(meta)

    # 2. Missing values
    print("Missing feature cells:", df.isna().sum().sum())
    print("Missing metadata cells:", meta.isna().sum().sum())

    assert df.isna().sum().sum() == 0
    assert meta.isna().sum().sum() == 0

    # 3. Length validation
    assert df["length"].gt(0).all()

    print("Minimum length:", df["length"].min())

    # 4. Duplicate sample validation
    keys = ["segment_id", "observation_time_utc"]

    assert not meta.duplicated(keys).any()

    print("Duplicate sample keys:", meta.duplicated(keys).sum())

    # 5. Target interval validation
    observation = pd.to_datetime(
        meta["observation_time_utc"], utc=True
    )

    target_time = pd.to_datetime(
        meta["target_time_utc"], utc=True
    )

    interval = (
        target_time - observation
    ).dt.total_seconds() / 60

    assert interval.eq(30).all()

    print("Target interval:", interval.unique())

    # 6. Target class distribution
    print("\nTarget distribution:")
    display(
        df["target_los_30m"].value_counts()
        .sort_index()
        .to_frame("Count")
    )

    print("Target classes:",
          sorted(df["target_los_30m"].unique()))


# TRAIN - HOLDOUT OVERLAP

train_keys = set(zip(
    train_meta_filtered["segment_id"],
    train_meta_filtered["observation_time_utc"]
))

holdout_keys = set(zip(
    holdout_meta_filtered["segment_id"],
    holdout_meta_filtered["observation_time_utc"]
))

overlap = train_keys.intersection(holdout_keys)

print("\nTRAIN-HOLDOUT OVERLAP")
print("Common sample keys:", len(overlap))

assert len(overlap) == 0

# TEMPORAL SEPARATION


latest_train_target = pd.to_datetime(
    train_meta_filtered["target_time_utc"], utc=True
).max()

earliest_holdout_observation = pd.to_datetime(
    holdout_meta_filtered["observation_time_utc"], utc=True
).min()

print("\nTEMPORAL SEPARATION")
print("Latest train target:", latest_train_target)
print("Earliest holdout observation:",
      earliest_holdout_observation)

assert latest_train_target < earliest_holdout_observation

print("\nALL FINAL INTEGRITY CHECKS PASSED!")



[TRAIN INTEGRITY AUDIT]

Feature shape: (4879, 20)
Metadata shape: (4879, 4)
Missing feature cells: 0
Missing metadata cells: 0
Minimum length: 1
Duplicate sample keys: 0
Target interval: [30.]

Target distribution:


,Count
target_los_30m,
A,956
B,153
C,803
D,1701
E,1071
F,195


Target classes: ['A', 'B', 'C', 'D', 'E', 'F']

[HOLDOUT INTEGRITY AUDIT]

Feature shape: (2998, 20)
Metadata shape: (2998, 4)
Missing feature cells: 0
Missing metadata cells: 0
Minimum length: 1
Duplicate sample keys: 0
Target interval: [30.]

Target distribution:


,Count
target_los_30m,
A,1635
B,1004
C,32
D,35
E,24
F,268


Target classes: ['A', 'B', 'C', 'D', 'E', 'F']

TRAIN-HOLDOUT OVERLAP
Common sample keys: 0

TEMPORAL SEPARATION
Latest train target: 2020-12-06 15:00:00+00:00
Earliest holdout observation: 2020-12-07 09:00:00+00:00

ALL FINAL INTEGRITY CHECKS PASSED!


In [22]:

# FEATURE - METADATA ALIGNMENT AUDIT

assert train_filtered.index.equals(train_meta_filtered.index)
assert holdout_filtered.index.equals(holdout_meta_filtered.index)

assert len(train_filtered) == len(train_meta_filtered)
assert len(holdout_filtered) == len(holdout_meta_filtered)

print("TRAIN")
print("Feature index equals metadata index:",
      train_filtered.index.equals(train_meta_filtered.index))

print("\nHOLDOUT")
print("Feature index equals metadata index:",
      holdout_filtered.index.equals(holdout_meta_filtered.index))

print("\nTRAIN METADATA COLUMNS")
print(train_meta_filtered.columns.tolist())

print("\nHOLDOUT METADATA COLUMNS")
print(holdout_meta_filtered.columns.tolist())

print("\nTRAIN METADATA SAMPLE")
display(train_meta_filtered.head())

print("\nHOLDOUT METADATA SAMPLE")
display(holdout_meta_filtered.head())

print("\nFEATURE-METADATA ALIGNMENT PASSED!")


TRAIN
Feature index equals metadata index: True

HOLDOUT
Feature index equals metadata index: True

TRAIN METADATA COLUMNS
['segment_id', 'observation_time_local', 'observation_time_utc', 'target_time_utc']

HOLDOUT METADATA COLUMNS
['segment_id', 'observation_time_local', 'observation_time_utc', 'target_time_utc']

TRAIN METADATA SAMPLE


,segment_id,observation_time_local,observation_time_utc,target_time_utc
0,33824,2020-07-04 15:00:00+07:00,2020-07-04 08:00:00+00:00,2020-07-04 08:30:00+00:00
1,56816,2020-07-04 15:30:00+07:00,2020-07-04 08:30:00+00:00,2020-07-04 09:00:00+00:00
2,56816,2020-07-04 16:00:00+07:00,2020-07-04 09:00:00+00:00,2020-07-04 09:30:00+00:00
3,56816,2020-07-04 16:30:00+07:00,2020-07-04 09:30:00+00:00,2020-07-04 10:00:00+00:00
4,56816,2020-07-04 20:30:00+07:00,2020-07-04 13:30:00+00:00,2020-07-04 14:00:00+00:00



HOLDOUT METADATA SAMPLE


,segment_id,observation_time_local,observation_time_utc,target_time_utc
0,84474,2020-12-07 16:00:00+07:00,2020-12-07 09:00:00+00:00,2020-12-07 09:30:00+00:00
1,6121,2020-12-07 20:00:00+07:00,2020-12-07 13:00:00+00:00,2020-12-07 13:30:00+00:00
2,39050,2020-12-07 20:00:00+07:00,2020-12-07 13:00:00+00:00,2020-12-07 13:30:00+00:00
3,21967,2020-12-07 20:00:00+07:00,2020-12-07 13:00:00+00:00,2020-12-07 13:30:00+00:00
4,21969,2020-12-07 20:00:00+07:00,2020-12-07 13:00:00+00:00,2020-12-07 13:30:00+00:00



FEATURE-METADATA ALIGNMENT PASSED!


## 17. Export the final V2 dataset

Export exactly one V2 release to `data/processed/model_v2/`. The project root detected in Section 1 is reused, so the output path is independent of whether the notebook is launched from the repository root or the `notebooks/` directory.

The feature tables and metadata are created from the same filtered row order. Identifiers and timestamps stay in metadata, not in the feature table.


In [23]:

# FINAL DATASET EXPORT - MODEL V2


OUTPUT_DIR = PROJECT_ROOT / "data" / "processed" / "model_v2"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Final feature tables
train_dataset_v2 = train_filtered.reset_index(drop=True).copy()
holdout_dataset_v2 = holdout_filtered.reset_index(drop=True).copy()

# Final metadata tables
train_metadata_v2 = train_meta_filtered.reset_index(drop=True).copy()
holdout_metadata_v2 = holdout_meta_filtered.reset_index(drop=True).copy()

# Add stable row order and sample identifier.
for metadata in [train_metadata_v2, holdout_metadata_v2]:
    observation_time = pd.to_datetime(
        metadata["observation_time_utc"], utc=True
    )

    metadata.insert(
        0,
        "row_order",
        range(len(metadata))
    )

    metadata.insert(
        1,
        "sample_id",
        metadata["segment_id"].astype(str)
        + "_"
        + observation_time.dt.strftime("%Y%m%dT%H%M%SZ")
    )

    assert metadata["sample_id"].is_unique

# Verify feature/metadata alignment before writing.
assert len(train_dataset_v2) == len(train_metadata_v2)
assert len(holdout_dataset_v2) == len(holdout_metadata_v2)

assert train_dataset_v2.index.equals(train_metadata_v2.index)
assert holdout_dataset_v2.index.equals(holdout_metadata_v2.index)

# Verify exact target horizon.
for metadata in [train_metadata_v2, holdout_metadata_v2]:
    observation_time = pd.to_datetime(
        metadata["observation_time_utc"], utc=True
    )
    target_time = pd.to_datetime(
        metadata["target_time_utc"], utc=True
    )

    assert (
        (target_time - observation_time).dt.total_seconds()
        == 30 * 60
    ).all()

# Write the four Parquet deliverables.
train_dataset_v2.to_parquet(
    OUTPUT_DIR / "train_dataset_v2.parquet",
    index=False
)

holdout_dataset_v2.to_parquet(
    OUTPUT_DIR / "holdout_dataset_v2.parquet",
    index=False
)

train_metadata_v2.to_parquet(
    OUTPUT_DIR / "train_metadata_v2.parquet",
    index=False
)

holdout_metadata_v2.to_parquet(
    OUTPUT_DIR / "holdout_metadata_v2.parquet",
    index=False
)

print("V2 DATASETS EXPORTED")
print("Output directory:", OUTPUT_DIR.resolve())
print("Train:", train_dataset_v2.shape)
print("Holdout:", holdout_dataset_v2.shape)
print("Train metadata:", train_metadata_v2.shape)
print("Holdout metadata:", holdout_metadata_v2.shape)


V2 DATASETS EXPORTED
Output directory: C:\CODE_PROJECTS\KHAI PHA DU LIEU\FinalTermProject\hcmc-traffic-los-forecasting\data\processed\model_v2
Train: (4879, 20)
Holdout: (2998, 20)
Train metadata: (4879, 6)
Holdout metadata: (2998, 6)


## 18. Validate the Parquet round trip

Reload the exported files and confirm that shapes, columns, row counts, identifiers, and missing-value checks are preserved.


In [24]:

# PARQUET ROUND-TRIP VALIDATION

train_check = pd.read_parquet(
    OUTPUT_DIR / "train_dataset_v2.parquet"
)
holdout_check = pd.read_parquet(
    OUTPUT_DIR / "holdout_dataset_v2.parquet"
)
train_meta_check = pd.read_parquet(
    OUTPUT_DIR / "train_metadata_v2.parquet"
)
holdout_meta_check = pd.read_parquet(
    OUTPUT_DIR / "holdout_metadata_v2.parquet"
)

# Shapes and columns
assert train_check.shape == train_dataset_v2.shape
assert holdout_check.shape == holdout_dataset_v2.shape

assert train_meta_check.shape == train_metadata_v2.shape
assert holdout_meta_check.shape == holdout_metadata_v2.shape

assert list(train_check.columns) == list(holdout_check.columns)
assert list(train_meta_check.columns) == list(holdout_meta_check.columns)

# Missing values
assert train_check.isna().sum().sum() == 0
assert holdout_check.isna().sum().sum() == 0
assert train_meta_check.isna().sum().sum() == 0
assert holdout_meta_check.isna().sum().sum() == 0

# Identifier and row-order integrity
assert train_meta_check["sample_id"].is_unique
assert holdout_meta_check["sample_id"].is_unique

assert train_meta_check["row_order"].tolist() == list(
    range(len(train_meta_check))
)
assert holdout_meta_check["row_order"].tolist() == list(
    range(len(holdout_meta_check))
)

assert (
    train_meta_check["sample_id"].astype(str).tolist()
    == train_metadata_v2["sample_id"].astype(str).tolist()
)
assert (
    holdout_meta_check["sample_id"].astype(str).tolist()
    == holdout_metadata_v2["sample_id"].astype(str).tolist()
)

print("PARQUET ROUND-TRIP VALIDATION PASSED!")
print("Train:", train_check.shape)
print("Holdout:", holdout_check.shape)
print("Train metadata:", train_meta_check.shape)
print("Holdout metadata:", holdout_meta_check.shape)

print("\nEXPORTED PARQUET FILES")
for filename in [
    "train_dataset_v2.parquet",
    "holdout_dataset_v2.parquet",
    "train_metadata_v2.parquet",
    "holdout_metadata_v2.parquet"
]:
    file_path = OUTPUT_DIR / filename
    print(
        f"{filename} - "
        f"{file_path.stat().st_size / 1024:.2f} KB"
    )


PARQUET ROUND-TRIP VALIDATION PASSED!
Train: (4879, 20)
Holdout: (2998, 20)
Train metadata: (4879, 6)
Holdout metadata: (2998, 6)

EXPORTED PARQUET FILES
train_dataset_v2.parquet - 85.93 KB
holdout_dataset_v2.parquet - 44.95 KB
train_metadata_v2.parquet - 92.51 KB
holdout_metadata_v2.parquet - 57.94 KB


## 19. Create the V2 feature schema

Write a machine-readable description of the target, feature types, observed training categories, dataset sizes, temporal split, metadata, preprocessing decisions, and known limitations.


In [25]:

# FEATURE SCHEMA - MODEL V2

import json

TARGET = TARGET_COLUMN
CLASSES = LOS_CLASSES

numeric_features = NUMERIC_FEATURES.copy()
categorical_features = CATEGORICAL_FEATURES.copy()

feature_columns = [
    column for column in train_check.columns
    if column != TARGET
]

assert set(numeric_features).isdisjoint(categorical_features)
assert set(numeric_features + categorical_features) == set(feature_columns)
assert len(feature_columns) == 19
assert TARGET in train_check.columns
assert list(train_check.columns) == list(holdout_check.columns)

def get_time_range(metadata, column):
    values = pd.to_datetime(metadata[column], utc=True)
    return {
        "start": values.min().isoformat(),
        "end": values.max().isoformat()
    }

def get_class_distribution(data):
    return {
        str(label): int(count)
        for label, count in (
            data[TARGET]
            .value_counts()
            .reindex(CLASSES, fill_value=0)
            .items()
        )
    }

category_values = {
    column: sorted(
        train_check[column].dropna().astype(str).unique().tolist()
    )
    for column in categorical_features
}

schema = {
    "project": "HCMC Traffic LOS Forecasting",
    "dataset_version": "2.0",
    "task": "Multiclass classification",
    "prediction_objective": "Predict LOS 30 minutes ahead",
    "prediction_horizon_minutes": 30,
    "timezone": "Asia/Ho_Chi_Minh",

    "target": {
        "name": TARGET,
        "classes": CLASSES,
        "label_source": "Source-provided LOS at future observation",
        "generation_rule": (
            "Exact match on the same segment_id at "
            "observation_time_utc + 30 minutes"
        ),
        "label_definition_status": (
            "Formal LOS threshold documentation has not been verified"
        )
    },

    "features": {
        "total_features": len(feature_columns),
        "numeric_features": numeric_features,
        "categorical_features": categorical_features,
        "categorical_values_observed_in_train": category_values,
        "feature_columns": feature_columns
    },

    "datasets": {
        "source_observations": SOURCE_OBSERVATION_COUNT,
        "valid_targets_before_road_filter": (
            VALID_TARGET_COUNT_BEFORE_ROAD_FILTER
        ),
        "target_coverage_before_road_filter_percent": round(
            TARGET_COVERAGE_PERCENT, 2
        ),
        "observations_removed_by_source_length_filter": len(excluded_samples),
        "train": {
            "file": "train_dataset_v2.parquet",
            "samples": len(train_check),
            "target_distribution": get_class_distribution(train_check),
            "observation_time_utc": get_time_range(
                train_meta_check, "observation_time_utc"
            )
        },
        "holdout": {
            "file": "holdout_dataset_v2.parquet",
            "samples": len(holdout_check),
            "target_distribution": get_class_distribution(holdout_check),
            "observation_time_utc": get_time_range(
                holdout_meta_check, "observation_time_utc"
            )
        }
    },

    "metadata": {
        "train_file": "train_metadata_v2.parquet",
        "holdout_file": "holdout_metadata_v2.parquet",
        "columns": train_meta_check.columns.tolist(),
        "purpose": "Sample identification and temporal traceability"
    },

    "split_strategy": {
        "method": "Chronological holdout",
        "latest_train_target_utc": (
            pd.to_datetime(
                train_meta_check["target_time_utc"], utc=True
            ).max().isoformat()
        ),
        "earliest_holdout_observation_utc": (
            pd.to_datetime(
                holdout_meta_check["observation_time_utc"], utc=True
            ).min().isoformat()
        )
    },

    "preprocessing": {
        "missing_values": "No missing values in final feature and metadata tables",
        "duplicate_samples": "No duplicate segment-time sample keys",
        "source_length_filter": (
            "Exclude observations mapped to segments whose source length is 0 m. "
            "The coordinate audit found that source length matches the floored "
            "Haversine endpoint distance; these records are sub-meter after "
            "source rounding and are not asserted to be malformed road geometry."
        ),
        "categorical_encoding": "Not applied",
        "numerical_scaling": "Not applied",
        "class_balancing": "Not applied"
    },

    "quality_validation": {
        "parquet_round_trip": True,
        "feature_metadata_row_alignment": True,
        "unique_sample_ids": True,
        "target_interval_30_minutes": True,
        "train_holdout_temporal_separation": True,
        "missing_values": 0
    },

    "known_limitations": [
        "Valid future targets cover 23.62% of original traffic observations before the source-length filter.",
        "A strong LOS distribution shift exists between Train and chronological Holdout.",
        "Formal source LOS label thresholds have not been verified.",
        "Weather variables represent the historical weather source location, not road-level weather stations.",
        "Source-zero-length segments are sub-meter according to the endpoint-coordinate audit; the exclusion is a documented dataset policy."
    ]
}

schema_path = OUTPUT_DIR / "feature_schema_v2.json"

with open(schema_path, "w", encoding="utf-8") as file:
    json.dump(schema, file, indent=4, ensure_ascii=False)

print("FEATURE SCHEMA CREATED")
print("Path:", schema_path.resolve())
print("Total features:", schema["features"]["total_features"])
print("Numeric:", len(numeric_features))
print("Categorical:", len(categorical_features))
print("Train samples:", schema["datasets"]["train"]["samples"])
print("Holdout samples:", schema["datasets"]["holdout"]["samples"])


FEATURE SCHEMA CREATED
Path: C:\CODE_PROJECTS\KHAI PHA DU LIEU\FinalTermProject\hcmc-traffic-los-forecasting\data\processed\model_v2\feature_schema_v2.json
Total features: 19
Numeric: 15
Categorical: 4
Train samples: 4879
Holdout samples: 2998


## 20. Run the final V2 data-quality audit

The release checks include the expected V2 sample counts. These constants define this specific V2 release; if the source data or cleaning policy changes, create a new dataset version and update the release contract rather than silently changing V2.


In [26]:

# FINAL DATA QUALITY AUDIT - MODEL V2

import json

# V2 release contract
EXPECTED_TRAIN_SAMPLES = 4879
EXPECTED_HOLDOUT_SAMPLES = 2998
EXPECTED_FEATURE_COUNT = 19
EXPECTED_METADATA_COLUMNS = 6

# Reload the exported release.
train = pd.read_parquet(OUTPUT_DIR / "train_dataset_v2.parquet")
holdout = pd.read_parquet(OUTPUT_DIR / "holdout_dataset_v2.parquet")
train_meta = pd.read_parquet(OUTPUT_DIR / "train_metadata_v2.parquet")
holdout_meta = pd.read_parquet(OUTPUT_DIR / "holdout_metadata_v2.parquet")

with open(OUTPUT_DIR / "feature_schema_v2.json", encoding="utf-8") as file:
    schema = json.load(file)

TARGET = schema["target"]["name"]
CLASSES = schema["target"]["classes"]
NUMERIC = schema["features"]["numeric_features"]
CATEGORICAL = schema["features"]["categorical_features"]
FEATURES = schema["features"]["feature_columns"]

results = {}

def check(name, condition):
    results[name] = bool(condition)
    print(f"[{'PASS' if condition else 'FAIL'}] {name}")

def sample_keys(metadata):
    observation_time = pd.to_datetime(
        metadata["observation_time_utc"], utc=True
    )
    return (
        metadata["segment_id"].astype(str)
        + "_"
        + observation_time.dt.strftime("%Y%m%dT%H%M%SZ")
    )

def validate_metadata_identifiers(metadata):
    expected_id = sample_keys(metadata)
    return (
        metadata["row_order"].tolist() == list(range(len(metadata)))
        and metadata["sample_id"].is_unique
        and metadata["sample_id"].astype(str).tolist() == expected_id.astype(str).tolist()
    )

def validate_target_interval(metadata):
    observation_time = pd.to_datetime(
        metadata["observation_time_utc"], utc=True
    )
    target_time = pd.to_datetime(
        metadata["target_time_utc"], utc=True
    )
    return (
        (target_time - observation_time).dt.total_seconds()
        == 30 * 60
    ).all()


print("\n[FINAL DATA QUALITY AUDIT - MODEL V2]\n")


# 1. Core deliverables
core_files = [
    "train_dataset_v2.parquet",
    "holdout_dataset_v2.parquet",
    "train_metadata_v2.parquet",
    "holdout_metadata_v2.parquet",
    "feature_schema_v2.json"
]
check(
    "All 5 core deliverables exist",
    all((OUTPUT_DIR / filename).is_file() for filename in core_files)
)

# 2. Release dimensions
check(
    "Train release shape",
    train.shape == (
        EXPECTED_TRAIN_SAMPLES,
        EXPECTED_FEATURE_COUNT + 1
    )
)
check(
    "Holdout release shape",
    holdout.shape == (
        EXPECTED_HOLDOUT_SAMPLES,
        EXPECTED_FEATURE_COUNT + 1
    )
)
check(
    "Train metadata shape",
    train_meta.shape == (
        EXPECTED_TRAIN_SAMPLES,
        EXPECTED_METADATA_COLUMNS
    )
)
check(
    "Holdout metadata shape",
    holdout_meta.shape == (
        EXPECTED_HOLDOUT_SAMPLES,
        EXPECTED_METADATA_COLUMNS
    )
)

# 3. Feature schema
check(
    "Train and Holdout columns match",
    list(train.columns) == list(holdout.columns)
)
check(
    "Feature columns match schema",
    [column for column in train.columns if column != TARGET] == FEATURES
)
check(
    "Numerical and categorical features are complete and disjoint",
    set(NUMERIC).isdisjoint(CATEGORICAL)
    and set(NUMERIC + CATEGORICAL) == set(FEATURES)
    and len(FEATURES) == EXPECTED_FEATURE_COUNT
)
check("Target exists", TARGET in train.columns and TARGET in holdout.columns)

# 4. Missing and infinite values
check("No missing values in Train", train.isna().sum().sum() == 0)
check("No missing values in Holdout", holdout.isna().sum().sum() == 0)
check(
    "No missing metadata",
    train_meta.isna().sum().sum() == 0
    and holdout_meta.isna().sum().sum() == 0
)
check(
    "No infinite numerical values",
    np.isfinite(train[NUMERIC].to_numpy(dtype=float)).all()
    and np.isfinite(holdout[NUMERIC].to_numpy(dtype=float)).all()
)

# 5. Road-length policy
check(
    "All retained road lengths are positive",
    train["length"].gt(0).all()
    and holdout["length"].gt(0).all()
)

# 6. Duplicate and overlap checks
train_keys = sample_keys(train_meta)
holdout_keys = sample_keys(holdout_meta)

check("No duplicate Train samples", train_keys.is_unique)
check("No duplicate Holdout samples", holdout_keys.is_unique)
check(
    "No Train-Holdout sample overlap",
    set(train_keys).isdisjoint(set(holdout_keys))
)

# 7. Target classes
check(
    "Train contains all LOS classes",
    set(train[TARGET].dropna().unique()) == set(CLASSES)
)
check(
    "Holdout contains all LOS classes",
    set(holdout[TARGET].dropna().unique()) == set(CLASSES)
)

# 8. Metadata identifiers and target intervals
check(
    "Train metadata identifiers valid",
    validate_metadata_identifiers(train_meta)
)
check(
    "Holdout metadata identifiers valid",
    validate_metadata_identifiers(holdout_meta)
)
check(
    "Train target interval = 30 minutes",
    validate_target_interval(train_meta)
)
check(
    "Holdout target interval = 30 minutes",
    validate_target_interval(holdout_meta)
)

# 9. Temporal separation and cross-partition identifiers
latest_train_target = pd.to_datetime(
    train_meta["target_time_utc"], utc=True
).max()

earliest_holdout_observation = pd.to_datetime(
    holdout_meta["observation_time_utc"], utc=True
).min()

check(
    "Temporal separation",
    latest_train_target < earliest_holdout_observation
)
check(
    "No duplicate sample IDs across partitions",
    set(train_meta["sample_id"]).isdisjoint(
        set(holdout_meta["sample_id"])
    )
)

# 10. Final summary
passed = sum(results.values())
failed = len(results) - passed

print("\n" + "=" * 60)
print("FINAL AUDIT SUMMARY")
print("=" * 60)
print("Total checks:", len(results))
print("Passed:", passed)
print("Failed:", failed)
print("\nTrain:", train.shape)
print("Holdout:", holdout.shape)
print("Total samples:", len(train) + len(holdout))

print("\nTrain target distribution:")
print(train[TARGET].value_counts().reindex(CLASSES, fill_value=0))

print("\nHoldout target distribution:")
print(holdout[TARGET].value_counts().reindex(CLASSES, fill_value=0))

if failed:
    failed_checks = [name for name, passed_check in results.items() if not passed_check]
    raise AssertionError(f"Final data-quality audit failed: {failed_checks}")

print("\nALL FINAL DATA QUALITY CHECKS PASSED!")



[FINAL DATA QUALITY AUDIT - MODEL V2]

[PASS] All 5 core deliverables exist
[PASS] Train release shape
[PASS] Holdout release shape
[PASS] Train metadata shape
[PASS] Holdout metadata shape
[PASS] Train and Holdout columns match
[PASS] Feature columns match schema
[PASS] Numerical and categorical features are complete and disjoint
[PASS] Target exists
[PASS] No missing values in Train
[PASS] No missing values in Holdout
[PASS] No missing metadata
[PASS] No infinite numerical values
[PASS] All retained road lengths are positive
[PASS] No duplicate Train samples
[PASS] No duplicate Holdout samples
[PASS] No Train-Holdout sample overlap
[PASS] Train contains all LOS classes
[PASS] Holdout contains all LOS classes
[PASS] Train metadata identifiers valid
[PASS] Holdout metadata identifiers valid
[PASS] Train target interval = 30 minutes
[PASS] Holdout target interval = 30 minutes
[PASS] Temporal separation
[PASS] No duplicate sample IDs across partitions

FINAL AUDIT SUMMARY
Total checks: 

## 21. Generate the data-quality report

Create a concise Markdown report from the exported V2 data and the final audit results. The report records the source-length exclusion policy and the known limitations without claiming that the dataset is fully representative or that model performance has been established.


In [27]:

# GENERATE DATA QUALITY REPORT - MODEL V2


report_path = OUTPUT_DIR / "data_quality_report_v2.md"

train_dist = (
    train[TARGET]
    .value_counts()
    .reindex(CLASSES, fill_value=0)
)

holdout_dist = (
    holdout[TARGET]
    .value_counts()
    .reindex(CLASSES, fill_value=0)
)

distribution_table = (
    "| LOS | Train | Train (%) | Holdout | Holdout (%) |\n"
    "|---|---:|---:|---:|---:|\n"
)

for label in CLASSES:
    train_count = int(train_dist[label])
    holdout_count = int(holdout_dist[label])

    distribution_table += (
        f"| {label} | {train_count} | "
        f"{train_count / len(train) * 100:.2f}% | "
        f"{holdout_count} | "
        f"{holdout_count / len(holdout) * 100:.2f}% |\n"
    )

audit_table = "| Check | Result |\n|---|---|\n"
for check_name, passed_check in results.items():
    audit_table += (
        f"| {check_name} | "
        f"{'PASS' if passed_check else 'FAIL'} |\n"
    )

passed_count = sum(results.values())
total_count = len(results)

if passed_count != total_count:
    raise AssertionError(
        "The data-quality report will not be generated because "
        "one or more final audit checks failed."
    )

report = f"""# Data Quality Report — HCMC Traffic LOS Forecasting

## 1. Dataset overview

| Attribute | Value |
|---|---|
| Dataset version | V2 |
| Preparation objective | LOS at t+30 minutes |
| Train samples | {len(train):,} |
| Holdout samples | {len(holdout):,} |
| Total final samples | {len(train) + len(holdout):,} |
| Features | {len(FEATURES)} |
| Numerical features | {len(NUMERIC)} |
| Categorical features | {len(CATEGORICAL)} |
| Target | {TARGET} |
| Target classes | {", ".join(CLASSES)} |

## 2. Deliverables

- `train_dataset_v2.parquet`
- `holdout_dataset_v2.parquet`
- `train_metadata_v2.parquet`
- `holdout_metadata_v2.parquet`
- `feature_schema_v2.json`
- `data_quality_report_v2.md`

## 3. Target coverage

- Original traffic observations: {SOURCE_OBSERVATION_COUNT:,}
- Valid future targets before road-length filtering: {VALID_TARGET_COUNT_BEFORE_ROAD_FILTER:,}
- Target coverage before road-length filtering: {TARGET_COVERAGE_PERCENT:.2f}%
- Samples excluded by the source-length filter: {len(excluded_samples):,}
- Final samples: {len(train) + len(holdout):,}

## 4. Target distribution

{distribution_table}

## 5. Final data-quality validation

- Total checks: {total_count}
- Passed: {passed_count}
- Failed: {total_count - passed_count}

{audit_table}

## 6. Data-preparation operations

- Integrated traffic observations with road-segment attributes and node coordinates.
- Integrated historical weather using a backward as-of join with a 30-minute tolerance.
- Generated the future LOS label by exact same-segment matching at t+30 minutes.
- Audited target coverage and temporal distribution.
- Created a chronological Train/Holdout partition and checked target-time separation.
- Refined temporal, road, and weather feature columns.
- Audited categorical availability and numerical feature drift.
- Audited source road-length precision using endpoint coordinates.
- Excluded observations mapped to source segments with a recorded length of 0 m.
- Preserved separate metadata for sample identification and temporal traceability.
- Exported and reloaded the V2 Parquet deliverables.

## 7. Source-length filtering policy

The source road table contains {len(source_zero_segment_ids):,} segments whose recorded integer length is 0 m. The endpoint-coordinate audit found that source lengths match the floored Haversine distance for the supplied road table. Thus, these records are sub-meter according to the endpoint coordinates.

The V2 pipeline excludes observations mapped to these segments to avoid zero-valued length features. This is a documented dataset policy; it does not establish that the corresponding road geometries are malformed.

## 8. Known limitations

- Exact future LOS labels are available for {TARGET_COVERAGE_PERCENT:.2f}% of the original traffic observations before road-length filtering.
- LOS class proportions differ substantially between Train and chronological Holdout.
- Formal source LOS classification thresholds have not been independently verified.
- Weather variables come from the historical weather source location and are not road-level station measurements.
- The observed roads and historical period should not be assumed to represent all roads or all traffic conditions in Ho Chi Minh City.

## 9. Handoff notes

- The feature table contains the selected predictors and `target_los_30m`.
- Segment identifiers and timestamps are stored separately in the metadata files.
- `feature_schema_v2.json` defines the feature names and their numerical/categorical groups.
- Categorical encoding, numerical scaling, and class balancing have not been applied in this data-preparation pipeline.
- The chronological Holdout partition has been preserved unchanged.

## 10. Conclusion

The V2 release passed all {total_count} implemented final data-quality checks.

The exported dataset is structurally validated for handoff to the downstream modeling stage. These checks do not establish label correctness, statistical representativeness, or future model performance.

Report version: 2.0
"""

report_path.write_text(report, encoding="utf-8")

assert report_path.is_file()
assert report_path.stat().st_size > 0

print("DATA QUALITY REPORT CREATED")
print("Path:", report_path.resolve())
print("Audit checks:", passed_count, "/", total_count)
print("Report size:", report_path.stat().st_size, "bytes")


DATA QUALITY REPORT CREATED
Path: C:\CODE_PROJECTS\KHAI PHA DU LIEU\FinalTermProject\hcmc-traffic-los-forecasting\data\processed\model_v2\data_quality_report_v2.md
Audit checks: 25 / 25
Report size: 5143 bytes


## 22. Completion

The V2 data-preparation pipeline is complete when all cells run from a fresh kernel, the final audit passes, and the six deliverables are present in `data/processed/model_v2/`.

This notebook ends at the data-preparation handoff. Model training and model evaluation are outside its scope.
